<a href="https://colab.research.google.com/github/vinizpl/Disponibilidade-do-artefato-Banco-de-Dados/blob/main/Artigo_BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import io
import hashlib
import json
import requests
import pandas as pd

url = (
    "https://huggingface.co/datasets/"
    "SalmaneExploring/pad-ufes-20/"
    "resolve/main/metadata.csv?download=true"
)

resposta = requests.get(url, timeout=120)
resposta.raise_for_status()
conteudo = resposta.content

df = pd.read_csv(
    io.BytesIO(conteudo),
    dtype=str,
    keep_default_na=False
)

colunas_exigidas = {
    "patient_id", "lesion_id",
    "img_id", "fitspatrick"
}

assert colunas_exigidas.issubset(df.columns), (
    "O arquivo não apresenta as colunas esperadas."
)

def distintos_nao_vazios(coluna):
    valores = df[coluna].str.strip()
    return valores[valores.ne("")].nunique()

contagens = {
    "registros": len(df),
    "pacientes": distintos_nao_vazios("patient_id"),
    "lesoes": distintos_nao_vazios("lesion_id"),
    "imagens": distintos_nao_vazios("img_id")
}

publicado = {
    "registros": 2298,
    "pacientes": 1373,
    "lesoes": 1641,
    "imagens": 2298
}

pares = df[
    df["patient_id"].str.strip().ne("") &
    df["lesion_id"].str.strip().ne("")
][["lesion_id", "patient_id"]].drop_duplicates()

pacientes_por_lesao = (
    pares.groupby("lesion_id")["patient_id"].nunique()
)

relatorio = {
    "sha256_csv": hashlib.sha256(conteudo).hexdigest(),
    "n_colunas": len(df.columns),
    "colunas": df.columns.tolist(),
    "contagens_observadas": contagens,
    "contagens_publicadas": publicado,
    "coincidem_com_publicacao": contagens == publicado,
    "imagens_duplicadas": int(
        df["img_id"].duplicated().sum()
    ),
    "lesoes_com_multiplos_pacientes": int(
        (pacientes_por_lesao > 1).sum()
    ),
    "fitzpatrick_valores_originais": (
        df["fitspatrick"]
        .value_counts(dropna=False)
        .to_dict()
    ),
    "raca_cor_ibge_explicita": False
}

print(json.dumps(
    relatorio,
    ensure_ascii=False,
    indent=2
))


{
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "n_colunas": 26,
  "colunas": [
    "patient_id",
    "lesion_id",
    "smoke",
    "drink",
    "background_father",
    "background_mother",
    "age",
    "pesticide",
    "gender",
    "skin_cancer_history",
    "cancer_history",
    "has_piped_water",
    "has_sewage_system",
    "fitspatrick",
    "region",
    "diameter_1",
    "diameter_2",
    "diagnostic",
    "itch",
    "grew",
    "hurt",
    "changed",
    "bleed",
    "elevation",
    "img_id",
    "biopsed"
  ],
  "contagens_observadas": {
    "registros": 2298,
    "pacientes": 1373,
    "lesoes": 1641,
    "imagens": 2298
  },
  "contagens_publicadas": {
    "registros": 2298,
    "pacientes": 1373,
    "lesoes": 1641,
    "imagens": 2298
  },
  "coincidem_com_publicacao": true,
  "imagens_duplicadas": 0,
  "lesoes_com_multiplos_pacientes": 250,
  "fitzpatrick_valores_originais": {
    "2.0": 876,
    "": 804,
    "3.0": 392,
    "

In [ ]:

import subprocess

subprocess.run(
    [
        "bash", "-lc",
        "apt-get update -qq && "
        "DEBIAN_FRONTEND=noninteractive "
        "apt-get install -y -qq "
        "postgresql-16 postgresql-client-16"
    ],
    check=True
)

subprocess.run(
    ["pg_ctlcluster", "16", "main", "start"],
    check=False
)

versao = subprocess.check_output(
    [
        "su", "postgres", "-c",
        "psql -Atqc 'SHOW server_version'"
    ],
    text=True
).strip()

print("PostgreSQL:", versao)
assert versao.startswith("16."), (
    "A versão do PostgreSQL não é 16."
)

print("Ambiente PostgreSQL 16 verificado.")


PostgreSQL: 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)
Ambiente PostgreSQL 16 verificado.


In [ ]:

import pandas as pd
import json

# Usa o CSV que ja esta carregado no notebook.
dados = df.copy()

for coluna in dados.columns:
    dados[coluna] = dados[coluna].astype(str).str.strip()

# Apenas associacoes com os dois IDs preenchidos.
validos = dados[
    dados["patient_id"].ne("") &
    dados["lesion_id"].ne("")
]

pares = validos[
    ["patient_id", "lesion_id"]
].drop_duplicates()

# Identificadores de lesao usados por mais de um paciente.
pacientes_por_id = (
    pares.groupby("lesion_id")["patient_id"].nunique()
)

colisoes = pacientes_por_id[pacientes_por_id > 1]

# Verifica divergencias de diagnostico no mesmo
# par paciente-lesao, sem pressupor que sejam erros.
diagnosticos = validos[
    validos["diagnostic"].ne("")
]

divergencias = (
    diagnosticos.groupby(
        ["patient_id", "lesion_id"]
    )["diagnostic"].nunique()
)

# Diferencia campo vazio do marcador explicito UNK.
ausencia = {}
for coluna in dados.columns:
    valores = dados[coluna]
    ausencia[coluna] = {
        "vazio": int((valores == "").sum()),
        "UNK": int(
            (valores.str.upper() == "UNK").sum()
        )
    }

resultado = {
    "linhas": len(dados),
    "pacientes_distintos": dados["patient_id"].nunique(),
    "ids_lesao_distintos": dados["lesion_id"].nunique(),
    "pares_paciente_lesao_distintos": len(pares),
    "ids_lesao_reutilizados": len(colisoes),
    "distribuicao_pacientes_por_id": {
        str(k): int(v)
        for k, v in pacientes_por_id.value_counts().items()
    },
    "pares_com_diagnosticos_divergentes": int(
        (divergencias > 1).sum()
    ),
    "ids_vazios": {
        coluna: int((dados[coluna] == "").sum())
        for coluna in ["patient_id", "lesion_id", "img_id"]
    },
    "ausencia_por_campo": ausencia
}

print(json.dumps(
    resultado,
    indent=2,
    ensure_ascii=False
))

with open(
    "auditoria_pad_ufes20.json", "w", encoding="utf-8"
) as arquivo:
    json.dump(
        resultado, arquivo,
        ensure_ascii=False, indent=2
    )

print("\nRelatorio salvo: auditoria_pad_ufes20.json")


{
  "linhas": 2298,
  "pacientes_distintos": 1373,
  "ids_lesao_distintos": 1641,
  "pares_paciente_lesao_distintos": 1891,
  "ids_lesao_reutilizados": 250,
  "distribuicao_pacientes_por_id": {
    "1": 1391,
    "2": 250
  },
  "pares_com_diagnosticos_divergentes": 0,
  "ids_vazios": {
    "patient_id": 0,
    "lesion_id": 0,
    "img_id": 0
  },
  "ausencia_por_campo": {
    "patient_id": {
      "vazio": 0,
      "UNK": 0
    },
    "lesion_id": {
      "vazio": 0,
      "UNK": 0
    },
    "smoke": {
      "vazio": 804,
      "UNK": 0
    },
    "drink": {
      "vazio": 804,
      "UNK": 0
    },
    "background_father": {
      "vazio": 818,
      "UNK": 95
    },
    "background_mother": {
      "vazio": 822,
      "UNK": 84
    },
    "age": {
      "vazio": 0,
      "UNK": 0
    },
    "pesticide": {
      "vazio": 804,
      "UNK": 0
    },
    "gender": {
      "vazio": 804,
      "UNK": 0
    },
    "skin_cancer_history": {
      "vazio": 804,
      "UNK": 0
    },
    "can

In [ ]:

import hashlib
import json
import os
import re
import subprocess
from pathlib import Path

# ==========================================================
# EXPERIMENTO 1 — PAD-UFES-20
# Comparação de contagens entre PostgreSQL 16 e pandas
# ==========================================================

# 1. Verificar se o CSV original permanece o mesmo

sha256 = hashlib.sha256(conteudo).hexdigest()

sha256_esperado = (
    "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527"
)

assert sha256 == sha256_esperado, (
    "O conteúdo do CSV mudou em relação à análise anterior."
)

print("SHA-256 do CSV confirmado:", sha256)


# 2. Inicializar o PostgreSQL 16

subprocess.run(
    ["pg_ctlcluster", "16", "main", "start"],
    capture_output=True,
    text=True
)

comando = [
    "runuser", "-u", "postgres", "--",
    "psql",
    "-X",
    "-q",
    "-t",
    "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

verificacao = subprocess.run(
    comando,
    input="SHOW server_version;",
    text=True,
    capture_output=True
)

assert verificacao.returncode == 0, verificacao.stderr

versao = verificacao.stdout.strip()

assert versao.startswith("16."), (
    f"Versão incorreta do PostgreSQL: {versao}"
)

print("PostgreSQL confirmado:", versao)


# 3. Preservar os bytes originais do CSV

caminho = Path("/tmp/pad_ufes20_original.csv")

caminho.write_bytes(conteudo)

os.chmod(caminho, 0o644)

print("CSV preparado para importação.")


# 4. Criar a definição da tabela temporária

colunas = df.columns.tolist()

assert len(colunas) == 26, (
    f"Esperadas 26 colunas; encontradas {len(colunas)}."
)

assert all(
    re.fullmatch(r"[a-z][a-z0-9_]*", coluna)
    for coluna in colunas
), "Há nomes de coluna inesperados."

definicoes = ",\n".join(
    f'"{coluna}" TEXT'
    for coluna in colunas
)


# 5. Executar a importação e a auditoria no PostgreSQL
#
# IMPORTANTE:
# O comando \copy deve estar em uma única linha.
# A tabela é temporária e será descartada ao fim da sessão.

sql = rf"""
BEGIN;

CREATE TEMP TABLE pad_csv (
{definicoes}
);

\copy pad_csv FROM '{caminho}' WITH (FORMAT csv, HEADER true, NULL '\N')

SELECT json_build_object(

    'postgresql',
    current_setting('server_version'),

    'registros',
    (SELECT COUNT(*) FROM pad_csv),

    'pacientes',
    (
        SELECT COUNT(DISTINCT patient_id)
        FROM pad_csv
    ),

    'lesion_id_distintos',
    (
        SELECT COUNT(DISTINCT lesion_id)
        FROM pad_csv
    ),

    'pares_paciente_lesao',
    (
        SELECT COUNT(DISTINCT (patient_id, lesion_id))
        FROM pad_csv
    ),

    'ids_lesao_reutilizados',
    (
        SELECT COUNT(*)
        FROM (
            SELECT lesion_id
            FROM pad_csv
            GROUP BY lesion_id
            HAVING COUNT(DISTINCT patient_id) > 1
        ) AS conflitos
    ),

    'imagens',
    (
        SELECT COUNT(DISTINCT img_id)
        FROM pad_csv
    ),

    'fitzpatrick_vazios',
    (
        SELECT COUNT(*)
        FROM pad_csv
        WHERE BTRIM(fitspatrick) = ''
    ),

    'background_father_vazios',
    (
        SELECT COUNT(*)
        FROM pad_csv
        WHERE BTRIM(background_father) = ''
    ),

    'background_father_unk',
    (
        SELECT COUNT(*)
        FROM pad_csv
        WHERE UPPER(BTRIM(background_father)) = 'UNK'
    )

);

ROLLBACK;
"""

resultado = subprocess.run(
    comando,
    input=sql,
    text=True,
    capture_output=True
)

if resultado.returncode != 0:
    print("ERRO DO POSTGRESQL:")
    print(resultado.stderr)
    raise RuntimeError("Falha na execução SQL.")

linhas_json = [
    linha
    for linha in resultado.stdout.splitlines()
    if linha.strip().startswith("{")
]

assert len(linhas_json) == 1, (
    "A consulta não retornou exatamente um JSON:\n"
    + resultado.stdout
)

saida = json.loads(linhas_json[0])


# 6. Comparar com as contagens obtidas anteriormente

esperado = {
    "registros": 2298,
    "pacientes": 1373,
    "lesion_id_distintos": 1641,
    "pares_paciente_lesao": 1891,
    "ids_lesao_reutilizados": 250,
    "imagens": 2298,
    "fitzpatrick_vazios": 804,
    "background_father_vazios": 818,
    "background_father_unk": 95
}

divergencias = {
    chave: {
        "postgresql": saida[chave],
        "pandas_anterior": valor
    }
    for chave, valor in esperado.items()
    if saida[chave] != valor
}

concordancia = len(divergencias) == 0


# 7. Exibir os resultados e preservar a evidência

relatorio = {
    "experimento": "PAD-UFES-20 — SQL versus pandas",
    "sha256_csv": sha256,
    "resultados_postgresql": saida,
    "referencia_pandas": esperado,
    "divergencias": divergencias,
    "concordancia_integral": concordancia
}

print("\n" + "=" * 55)
print("RESULTADOS DO EXPERIMENTO")
print("=" * 55)

print(json.dumps(
    relatorio,
    ensure_ascii=False,
    indent=2
))

arquivo_saida = Path(
    "/content/auditoria_pad_postgresql16.json"
)

arquivo_saida.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

Path("/content/consulta_auditoria_pad.sql").write_text(
    sql,
    encoding="utf-8"
)

print("\nRelatórios salvos no Colab.")
print("JSON:", arquivo_saida)
print("SQL: /content/consulta_auditoria_pad.sql")

assert concordancia, (
    "Foram encontradas divergências entre SQL e pandas."
)

print("\nTESTE CONCLUÍDO: todas as contagens conferem!")


SHA-256 do CSV confirmado: 14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527
PostgreSQL confirmado: 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)
CSV preparado para importação.

RESULTADOS DO EXPERIMENTO
{
  "experimento": "PAD-UFES-20 — SQL versus pandas",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "resultados_postgresql": {
    "postgresql": "16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)",
    "registros": 2298,
    "pacientes": 1373,
    "lesion_id_distintos": 1641,
    "pares_paciente_lesao": 1891,
    "ids_lesao_reutilizados": 250,
    "imagens": 2298,
    "fitzpatrick_vazios": 804,
    "background_father_vazios": 818,
    "background_father_unk": 95
  },
  "referencia_pandas": {
    "registros": 2298,
    "pacientes": 1373,
    "lesion_id_distintos": 1641,
    "pares_paciente_lesao": 1891,
    "ids_lesao_reutilizados": 250,
    "imagens": 2298,
    "fitzpatrick_vazios": 804,
    "background_father_vazios": 818,
    "background_father_unk"

In [ ]:

import hashlib
import io
import json
import subprocess
import zipfile
from pathlib import Path
from google.colab import files

# Conferir o CSV original, ja carregado no notebook.
sha_csv = hashlib.sha256(conteudo).hexdigest()
assert sha_csv == (
    "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527"
), "O CSV nao corresponde ao experimento anterior."

# Enviar o pacote ZIP pelo navegador.
print("Selecione o pacote ZIP baixado.")
enviados = files.upload()

assert len(enviados) == 1, "Envie somente o ZIP do experimento."
nome_zip, bytes_zip = next(iter(enviados.items()))

sha_zip = hashlib.sha256(bytes_zip).hexdigest()
assert sha_zip == (
    "94b1e0aad5352870dc9b2b9e4aa9391fdcc012123472b96873deb3f17e385b0f"
), "O ZIP enviado nao corresponde ao pacote preparado."

pasta = Path("/tmp/experimento_derm_v21")
pasta.mkdir(exist_ok=True)

with zipfile.ZipFile(io.BytesIO(bytes_zip)) as pacote:
    pacote.extractall(pasta)

# Preservar o CSV original para a carga.
csv = Path("/tmp/pad_ufes20_original.csv")
csv.write_bytes(conteudo)
csv.chmod(0o644)

subprocess.run(
    ["pg_ctlcluster", "16", "main", "start"],
    capture_output=True,
    text=True
)

psql = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-A", "-t",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

# Nao sobrescrever nenhum esquema existente.
checagem = subprocess.run(
    psql,
    input=(
        "SELECT current_setting('server_version');"
        "SELECT COUNT(*) FROM pg_namespace "
        "WHERE nspname='derm_v2';"
    ),
    text=True,
    capture_output=True
)

assert checagem.returncode == 0, checagem.stderr
linhas = checagem.stdout.strip().splitlines()

assert linhas[0].startswith("16."), (
    "PostgreSQL 16 nao identificado."
)

assert linhas[-1] == "0", (
    "O esquema derm_v2 ja existe. "
    "Nao o sobrescreva; envie o resultado para analise."
)

etapas = [
    "01_esquema_derm_v21_pg16.sql",
    "02_ingestao_pad_ufes20.sql",
    "03_teste_indice_composto.sql",
    "04_testes_restricoes_ah_pg16.sql"
]

logs = {}
resultado_carga = None

for nome in etapas:
    print("\nExecutando:", nome)

    processo = subprocess.run(
        psql + ["-f", str(pasta / nome)],
        text=True,
        capture_output=True
    )

    logs[nome] = {
        "codigo_retorno": processo.returncode,
        "stdout": processo.stdout,
        "stderr": processo.stderr
    }

    if processo.returncode != 0:
        print(processo.stderr)
        Path("/content/log_experimento_derm.json").write_text(
            json.dumps(logs, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )
        raise RuntimeError(
            f"Falha na etapa: {nome}. Nao execute "
            "novamente antes de verificar o erro."
        )

    if nome.startswith("02_"):
        candidatos = [
            linha for linha in processo.stdout.splitlines()
            if linha.strip().startswith("{")
        ]
        assert len(candidatos) == 1, processo.stdout
        resultado_carga = json.loads(candidatos[0])

    print("Etapa executada sem erro SQL.")

Path("/content/log_experimento_derm.json").write_text(
    json.dumps(logs, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nRESULTADO DA INGESTAO:")
print(json.dumps(
    resultado_carga,
    indent=2,
    ensure_ascii=False
))

print("\nArquivo de evidencias:")
print("/content/log_experimento_derm.json")


Selecione o pacote ZIP baixado.


Saving pad_ufes20_derm_v21_experimento.zip to pad_ufes20_derm_v21_experimento.zip

Executando: 01_esquema_derm_v21_pg16.sql
Etapa executada sem erro SQL.

Executando: 02_ingestao_pad_ufes20.sql
Etapa executada sem erro SQL.

Executando: 03_teste_indice_composto.sql
Etapa executada sem erro SQL.

Executando: 04_testes_restricoes_ah_pg16.sql
Etapa executada sem erro SQL.

RESULTADO DA INGESTAO:
{
  "fonte_id": 1,
  "staging": 2298,
  "pacientes": 1373,
  "lesoes_com_identidade_composta": 1891,
  "imagens": 2298,
  "observacoes_fitzpatrick": 2298,
  "fitzpatrick_ausente": 804,
  "fitzpatrick_preenchido": 1494,
  "rejeicoes_vinculo": 0
}

Arquivo de evidencias:
/content/log_experimento_derm.json


In [ ]:
# EXPERIMENTO 3 — preservacao dos 26 campos e integridade dos vinculos.
# Execute no mesmo Google Colab, DEPOIS das quatro etapas do experimento v2.1.
import hashlib
import json
import os
import re
import subprocess
from pathlib import Path

sha = hashlib.sha256(conteudo).hexdigest()
assert sha == '14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527', 'CSV diferente'
assert len(df.columns) == 26
assert all(re.fullmatch(r'[a-z][a-z0-9_]*', c) for c in df.columns)

arquivo_csv = Path('/tmp/pad_ufes20_original.csv')
arquivo_csv.write_bytes(conteudo)
os.chmod(arquivo_csv, 0o644)

definicoes = ',\n'.join(f'"{c}" text' for c in df.columns)

sql = rf"""
BEGIN;
CREATE TEMP TABLE pad_csv (
{definicoes}
) ON COMMIT DROP;
\copy pad_csv FROM '{arquivo_csv}' WITH (FORMAT csv, HEADER true, NULL '\N')
WITH f AS (
    SELECT fonte_id FROM derm_v2.fonte
    WHERE nome = 'PAD-UFES-20 metadata.csv'
      AND versao = 'sha256:{sha}'
), conferencia AS (
    SELECT
        COUNT(*) AS linhas_comparadas,
        COUNT(*) FILTER (
            WHERE s.staging_id IS NOT NULL
              AND s.registro_jsonb = to_jsonb(r)
        ) AS linhas_26_campos_preservados,
        COUNT(*) FILTER (
            WHERE p.paciente_id IS NOT NULL
              AND l.lesao_id IS NOT NULL
              AND i.imagem_id IS NOT NULL
              AND i.staging_id = s.staging_id
        ) AS linhas_vinculos_corretos,
        COUNT(*) FILTER (
            WHERE o.observacao_id IS NOT NULL
              AND o.valor_origem = r.fitspatrick
              AND o.staging_id = s.staging_id
              AND o.estado = CASE WHEN r.fitspatrick = ''
                    THEN 'ausente'::derm_v2.estado_observacao
                    ELSE 'preenchido'::derm_v2.estado_observacao
                  END
              AND m.mapeamento_id IS NOT NULL
              AND m.valor_origem = r.fitspatrick
              AND m.estado_destino = o.estado
              AND m.valor_categoria_destino
                  IS NOT DISTINCT FROM o.valor_categoria
        ) AS linhas_fitzpatrick_rastreaveis
    FROM pad_csv r
    CROSS JOIN f
    LEFT JOIN derm_v2.staging_registro s
      ON s.fonte_id = f.fonte_id
     AND s.identificador_registro_origem = r.img_id
    LEFT JOIN derm_v2.paciente p
      ON p.fonte_id = f.fonte_id
     AND p.identificador_origem = r.patient_id
    LEFT JOIN derm_v2.lesao l
      ON l.fonte_id = f.fonte_id
     AND l.paciente_id = p.paciente_id
     AND l.identificador_origem = r.lesion_id
    LEFT JOIN derm_v2.imagem i
      ON i.fonte_id = f.fonte_id
     AND i.identificador_origem = r.img_id
     AND i.lesao_id = l.lesao_id
    LEFT JOIN derm_v2.observacao o
      ON o.fonte_id = f.fonte_id
     AND o.imagem_id = i.imagem_id
     AND o.atributo = 'fitzpatrick'
     AND o.unidade = 'imagem'
    LEFT JOIN derm_v2.mapeamento_valor m
      ON m.mapeamento_id = o.mapeamento_id
     AND m.fonte_id = f.fonte_id
)
SELECT row_to_json(conferencia) FROM conferencia;
ROLLBACK;
"""

psql = [
    'runuser', '-u', 'postgres', '--',
    'psql', '-X', '-q', '-t', '-A',
    '-v', 'ON_ERROR_STOP=1', '-d', 'postgres'
]
r = subprocess.run(psql, input=sql, text=True, capture_output=True)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError('Falha no teste de integridade')
linhas_json = [s for s in r.stdout.splitlines() if s.strip().startswith('{')]
assert len(linhas_json) == 1, r.stdout
medidas = json.loads(linhas_json[0])

esperado = len(df)
criterios = {
    nome: (valor == esperado) for nome, valor in medidas.items()
}
relatorio = {
    'experimento': 'Confronto linha a linha: CSV original x derm_v2',
    'sha256_csv': sha,
    'versao_postgresql_requerida': '16.x',
    'resultados': medidas,
    'criterios': criterios,
    'todos_atendidos': all(criterios.values()),
    'nota': 'Comparacao de campos estruturados de texto/JSONB, nao igualdade binaria de CSV.'
}
saida = Path('/content/evidencia_integridade_26campos.json')
saida.write_text(json.dumps(relatorio, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(relatorio, indent=2, ensure_ascii=False))
print('Relatorio salvo em', saida)
assert relatorio['todos_atendidos'], 'Divergencias encontradas: veja o relatorio'


{
  "experimento": "Confronto linha a linha: CSV original x derm_v2",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "versao_postgresql_requerida": "16.x",
  "resultados": {
    "linhas_comparadas": 2298,
    "linhas_26_campos_preservados": 2298,
    "linhas_vinculos_corretos": 2298,
    "linhas_fitzpatrick_rastreaveis": 2298
  },
  "criterios": {
    "linhas_comparadas": true,
    "linhas_26_campos_preservados": true,
    "linhas_vinculos_corretos": true,
    "linhas_fitzpatrick_rastreaveis": true
  },
  "todos_atendidos": true,
  "nota": "Comparacao de campos estruturados de texto/JSONB, nao igualdade binaria de CSV."
}
Relatorio salvo em /content/evidencia_integridade_26campos.json


In [ ]:

import hashlib
import json
import subprocess
from pathlib import Path

# =====================================================
# AUDITORIA PAD-UFES-20 — SEM ZIP
# PostgreSQL 16 / esquema derm_v2
# =====================================================

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA
assert len(df) == 2298

psql = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1", "-d", "postgres"
]

def executar(sql):
    consulta = (
        "SELECT COALESCE(json_agg(row_to_json(t)), "
        "'[]'::json) FROM ("
        + sql.strip().rstrip(";")
        + ") AS t;"
    )

    r = subprocess.run(
        psql,
        input=consulta,
        capture_output=True,
        text=True
    )

    if r.returncode != 0:
        raise RuntimeError(r.stderr)

    return json.loads(r.stdout.strip())


# Confirmar a fonte exata no banco.
fontes = executar("""
    SELECT fonte_id, nome, versao
    FROM derm_v2.fonte
    WHERE nome = 'PAD-UFES-20 metadata.csv'
""")

assert len(fontes) == 1
assert fontes[0]["versao"] == "sha256:" + SHA

f = int(fontes[0]["fonte_id"])

print("Fonte identificada:", f)


# =====================================================
# Q1 — COBERTURA DE FITZPATRICK
# =====================================================

q1 = executar(f"""
    SELECT
        (
            SELECT COUNT(*)
            FROM derm_v2.imagem
            WHERE fonte_id = {f}
        )::int AS total_imagens,

        (
            SELECT COUNT(*)
            FROM derm_v2.observacao
            WHERE fonte_id = {f}
              AND atributo = 'fitzpatrick'
              AND unidade = 'imagem'
              AND estado = 'preenchido'
        )::int AS fitz_preenchido,

        (
            SELECT COUNT(*)
            FROM derm_v2.observacao
            WHERE fonte_id = {f}
              AND atributo = 'fitzpatrick'
              AND unidade = 'imagem'
              AND estado = 'ausente'
        )::int AS fitz_ausente,

        (
            SELECT COUNT(*)
            FROM derm_v2.catalogo_fonte_atributo
            WHERE fonte_id = {f}
              AND unidade = 'imagem'
              AND campo_existe IS FALSE
        )::int AS atributos_inexistentes
""")[0]


# =====================================================
# Q2 — DISTRIBUICAO DOS FOTOTIPOS
# =====================================================

q2 = executar(f"""
    SELECT
        CASE
            WHEN estado = 'preenchido'
                THEN valor_categoria
            ELSE '__' || estado::text || '__'
        END AS grupo,
        COUNT(*)::int AS n
    FROM derm_v2.observacao
    WHERE fonte_id = {f}
      AND atributo = 'fitzpatrick'
      AND unidade = 'imagem'
    GROUP BY 1
    ORDER BY 1
""")


# =====================================================
# Q3 — ESTADOS DE AUSENCIA
# =====================================================

q3 = executar(f"""
    SELECT
        'observacao'::text AS nivel,
        estado::text AS estado,
        COUNT(*)::int AS n
    FROM derm_v2.observacao
    WHERE fonte_id = {f}
    GROUP BY estado

    UNION ALL

    SELECT
        'catalogo',
        'campo_inexistente',
        COUNT(*)::int
    FROM derm_v2.catalogo_fonte_atributo
    WHERE fonte_id = {f}
      AND campo_existe IS FALSE

    ORDER BY nivel, estado
""")


# =====================================================
# Q4 — CONTAGEM DAS UNIDADES
# =====================================================

q4 = executar(f"""
    SELECT
        (
            SELECT COUNT(*)
            FROM derm_v2.paciente
            WHERE fonte_id = {f}
        )::int AS pacientes,

        (
            SELECT COUNT(*)
            FROM derm_v2.lesao
            WHERE fonte_id = {f}
        )::int AS lesoes,

        (
            SELECT COUNT(*)
            FROM derm_v2.imagem
            WHERE fonte_id = {f}
        )::int AS imagens
""")[0]


# =====================================================
# Q5 — DISPONIBILIDADE DOS DADOS CLINICOS
# Nao calcula contagem clinica sem pre-requisitos.
# =====================================================

q5 = executar(f"""
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM derm_v2.paciente
            WHERE fonte_id = {f}
              AND uf_residencia_ibge IS NOT NULL
        )
        THEN 'residencia_nao_documentada'

        WHEN NOT EXISTS (
            SELECT 1
            FROM derm_v2.diagnostico
            WHERE fonte_id = {f}
        )
        THEN 'sem_diagnostico_carregado'

        ELSE 'prerequisitos_disponiveis'
    END AS status
""")[0]


# =====================================================
# Q6 — VERIFICACAO DE VINCULOS
# =====================================================

q6 = executar(f"""
    SELECT
        (
            SELECT COUNT(*)
            FROM derm_v2.imagem
            WHERE fonte_id = {f}
              AND lesao_id IS NULL
        )::int AS imagens_sem_lesao,

        (
            SELECT COUNT(*)
            FROM derm_v2.lesao
            WHERE fonte_id = {f}
              AND paciente_id IS NULL
        )::int AS lesoes_sem_paciente
""")[0]


# =====================================================
# Q7 — DIVERGENCIAS ENTRE AVALIACOES
# =====================================================

q7 = executar(f"""
    SELECT
        i.identificador_origem AS img_id,
        o.atributo::text AS atributo,

        COUNT(
            DISTINCT COALESCE(
                o.valor_categoria,
                o.valor_numerico::text
            )
        )::int AS valores_distintos

    FROM derm_v2.observacao o

    JOIN derm_v2.imagem i
      ON i.imagem_id = o.imagem_id
     AND i.fonte_id = o.fonte_id

    WHERE o.fonte_id = {f}
      AND o.unidade = 'imagem'
      AND o.estado = 'preenchido'

    GROUP BY
        i.imagem_id,
        i.identificador_origem,
        o.atributo

    HAVING COUNT(
        DISTINCT COALESCE(
            o.valor_categoria,
            o.valor_numerico::text
        )
    ) > 1
""")


# =====================================================
# COMPARACAO INDEPENDENTE COM PANDAS
# =====================================================

fitz = df["fitspatrick"].astype(str).str.strip()

mapa = {
    "": "__ausente__",
    "1.0": "I",
    "2.0": "II",
    "3.0": "III",
    "4.0": "IV",
    "5.0": "V",
    "6.0": "VI"
}

assert set(fitz.unique()) <= set(mapa)

referencia = {
    k: int(v)
    for k, v in fitz.map(mapa).value_counts().items()
}

observado = {
    r["grupo"]: r["n"]
    for r in q2
}

assert q1["total_imagens"] == df["img_id"].nunique()
assert q1["fitz_preenchido"] == int(fitz.ne("").sum())
assert q1["fitz_ausente"] == int(fitz.eq("").sum())
assert q1["atributos_inexistentes"] == 3

assert observado == referencia, (
    f"Q2 divergiu: {observado} x {referencia}"
)

estados = {
    (r["nivel"], r["estado"]): r["n"]
    for r in q3
}

assert estados[("observacao", "preenchido")] == int(
    fitz.ne("").sum()
)
assert estados[("observacao", "ausente")] == int(
    fitz.eq("").sum()
)
assert estados[("catalogo", "campo_inexistente")] == 3

assert q4 == {
    "pacientes": df["patient_id"].nunique(),
    "lesoes": len(
        df[["patient_id", "lesion_id"]].drop_duplicates()
    ),
    "imagens": df["img_id"].nunique()
}

assert q5["status"] == "residencia_nao_documentada"

assert q6 == {
    "imagens_sem_lesao": 0,
    "lesoes_sem_paciente": 0
}

assert len(q7) == 0, (
    "Divergencias observadas: necessaria investigacao."
)


# =====================================================
# RELATORIO FINAL
# =====================================================

relatorio = {
    "experimento": "Auditoria PAD-UFES-20 sem ZIP",
    "sha256_csv": SHA,
    "fonte_id": f,
    "Q1": q1,
    "Q2": observado,
    "Q3": q3,
    "Q4": q4,
    "Q5": {
        "status": q5["status"],
        "contagem_clinica": None
    },
    "Q6": q6,
    "Q7": {
        "divergencias_observadas": len(q7)
    },
    "verificacoes_aplicaveis": "atendidas",
    "limites": (
        "Q5 sem dados clinicos suficientes; "
        "Q7 sem teste de conflito injetado; "
        "Q8 nao executada."
    )
}

print("\n" + "=" * 55)
print("RESULTADOS DA AUDITORIA")
print("=" * 55)

print(json.dumps(
    relatorio,
    ensure_ascii=False,
    indent=2
))

saida = Path(
    "/content/evidencia_Q1_Q7_sem_zip.json"
)

saida.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nArquivo salvo:", saida)
print("Verificacoes aplicaveis concluidas.")


Fonte identificada: 1

RESULTADOS DA AUDITORIA
{
  "experimento": "Auditoria PAD-UFES-20 sem ZIP",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "fonte_id": 1,
  "Q1": {
    "total_imagens": 2298,
    "fitz_preenchido": 1494,
    "fitz_ausente": 804,
    "atributos_inexistentes": 3
  },
  "Q2": {
    "__ausente__": 804,
    "I": 153,
    "II": 876,
    "III": 392,
    "IV": 62,
    "V": 10,
    "VI": 1
  },
  "Q3": [
    {
      "nivel": "catalogo",
      "estado": "campo_inexistente",
      "n": 3
    },
    {
      "nivel": "observacao",
      "estado": "ausente",
      "n": 804
    },
    {
      "nivel": "observacao",
      "estado": "preenchido",
      "n": 1494
    }
  ],
  "Q4": {
    "pacientes": 1373,
    "lesoes": 1891,
    "imagens": 2298
  },
  "Q5": {
    "status": "residencia_nao_documentada",
    "contagem_clinica": null
  },
  "Q6": {
    "imagens_sem_lesao": 0,
    "lesoes_sem_paciente": 0
  },
  "Q7": {
    "divergencias_observa

In [ ]:

import json
import hashlib
import subprocess
from pathlib import Path

# ==================================================
# EXPERIMENTO Q7 — INJECAO CONTROLADA DE DIVERGENCIA
# PostgreSQL 16 | PAD-UFES-20 | derm_v2
# ==================================================

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

def executar_sql(sql):
    resultado = subprocess.run(
        PSQL,
        input=sql,
        capture_output=True,
        text=True
    )
    if resultado.returncode != 0:
        raise RuntimeError(resultado.stderr)
    return resultado.stdout

# Identificar exclusivamente a versao correta da fonte.
consulta_fonte = f"""
SELECT fonte_id
FROM derm_v2.fonte
WHERE nome = 'PAD-UFES-20 metadata.csv'
  AND versao = 'sha256:{SHA}';
"""

linhas = executar_sql(consulta_fonte).strip().splitlines()
assert len(linhas) == 1
fonte_id = int(linhas[0])

# Mesma logica de deteccao utilizada na auditoria Q7.
consulta_conflitos = f"""
SELECT COUNT(*)
FROM (
    SELECT o.imagem_id, o.atributo
    FROM derm_v2.observacao o
    JOIN derm_v2.imagem i
      ON i.imagem_id = o.imagem_id
     AND i.fonte_id = o.fonte_id
    WHERE o.fonte_id = {fonte_id}
      AND o.unidade = 'imagem'
      AND o.estado = 'preenchido'
    GROUP BY o.imagem_id, o.atributo
    HAVING COUNT(DISTINCT COALESCE(
        o.valor_categoria,
        o.valor_numerico::text
    )) > 1
) conflitos
"""

sql = f"""
BEGIN;

SELECT json_build_object(
    'fase', 'antes',
    'conflitos', ({consulta_conflitos}),
    'total_observacoes', (
        SELECT COUNT(*)
        FROM derm_v2.observacao
        WHERE fonte_id = {fonte_id}
    )
);

-- Escolhe uma observacao REAL preenchida.
-- Seleciona um mapeamento REAL de outro fototipo.
-- A nova observacao e apenas uma falha SINTETICA.
WITH amostra AS (
    SELECT
        fonte_id,
        imagem_id,
        atributo,
        unidade,
        valor_categoria
    FROM derm_v2.observacao
    WHERE fonte_id = {fonte_id}
      AND atributo = 'fitzpatrick'
      AND unidade = 'imagem'
      AND estado = 'preenchido'
    ORDER BY observacao_id
    LIMIT 1
),
alternativa AS (
    SELECT
        a.fonte_id,
        a.imagem_id,
        a.atributo,
        a.unidade,
        m.mapeamento_id,
        m.valor_origem,
        m.valor_categoria_destino
    FROM amostra a
    JOIN LATERAL (
        SELECT *
        FROM derm_v2.mapeamento_valor mv
        WHERE mv.fonte_id = a.fonte_id
          AND mv.atributo = a.atributo
          AND mv.unidade = a.unidade
          AND mv.estado_destino = 'preenchido'
          AND mv.valor_categoria_destino
              <> a.valor_categoria
        ORDER BY mv.mapeamento_id
        LIMIT 1
    ) m ON TRUE
),
inserida AS (
    INSERT INTO derm_v2.observacao (
        fonte_id,
        atributo,
        unidade,
        imagem_id,
        estado,
        valor_categoria,
        valor_origem,
        mapeamento_id,
        metodo,
        situacao_atribuidor
    )
    SELECT
        fonte_id,
        atributo,
        unidade,
        imagem_id,
        'preenchido',
        valor_categoria_destino,
        valor_origem,
        mapeamento_id,
        'nao_documentado',
        'nao_documentado'
    FROM alternativa
    RETURNING imagem_id, valor_categoria
)
SELECT json_build_object(
    'fase', 'insercao_sintetica',
    'observacoes_inseridas', (
        SELECT COUNT(*) FROM inserida
    ),
    'imagem_id', (
        SELECT imagem_id FROM inserida LIMIT 1
    ),
    'valor_sintetico', (
        SELECT valor_categoria FROM inserida LIMIT 1
    )
);

SELECT json_build_object(
    'fase', 'durante',
    'conflitos', ({consulta_conflitos}),
    'total_observacoes', (
        SELECT COUNT(*)
        FROM derm_v2.observacao
        WHERE fonte_id = {fonte_id}
    )
);

-- Desfaz a observacao sintetica.
ROLLBACK;

SELECT json_build_object(
    'fase', 'depois',
    'conflitos', ({consulta_conflitos}),
    'total_observacoes', (
        SELECT COUNT(*)
        FROM derm_v2.observacao
        WHERE fonte_id = {fonte_id}
    )
);
"""

# Executar todas as etapas na mesma sessao.
saida_bruta = executar_sql(sql)

resultados = [
    json.loads(linha)
    for linha in saida_bruta.splitlines()
    if linha.strip().startswith("{")
]

assert len(resultados) == 4, saida_bruta

antes, insercao, durante, depois = resultados

criterios = {
    "sem_conflito_inicial": antes["conflitos"] == 0,
    "uma_observacao_sintetica": (
        insercao["observacoes_inseridas"] == 1
    ),
    "conflito_detectado": durante["conflitos"] == 1,
    "observacoes_restauradas": (
        depois["total_observacoes"]
        == antes["total_observacoes"]
    ),
    "conflitos_restaurados": (
        depois["conflitos"] == antes["conflitos"]
    )
}

relatorio = {
    "experimento": "Q7 - deteccao de divergencia",
    "fonte_id": fonte_id,
    "sha256_csv": SHA,
    "tipo_teste": "falha sintetica com ROLLBACK",
    "resultados": resultados,
    "criterios": criterios,
    "todos_atendidos": all(criterios.values())
}

arquivo = Path(
    "/content/evidencia_Q7_injecao_rollback.json"
)

arquivo.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    relatorio,
    ensure_ascii=False,
    indent=2
))

print("\nRelatorio salvo:", arquivo)

assert relatorio["todos_atendidos"], (
    "Um ou mais criterios nao foram atendidos."
)

print("\nTESTE Q7 CONCLUIDO COM SUCESSO.")


{
  "experimento": "Q7 - deteccao de divergencia",
  "fonte_id": 1,
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "tipo_teste": "falha sintetica com ROLLBACK",
  "resultados": [
    {
      "fase": "antes",
      "conflitos": 0,
      "total_observacoes": 2298
    },
    {
      "fase": "insercao_sintetica",
      "observacoes_inseridas": 1,
      "imagem_id": 2,
      "valor_sintetico": "I"
    },
    {
      "fase": "durante",
      "conflitos": 1,
      "total_observacoes": 2299
    },
    {
      "fase": "depois",
      "conflitos": 0,
      "total_observacoes": 2298
    }
  ],
  "criterios": {
    "sem_conflito_inicial": true,
    "uma_observacao_sintetica": true,
    "conflito_detectado": true,
    "observacoes_restauradas": true,
    "conflitos_restaurados": true
  },
  "todos_atendidos": true
}

Relatorio salvo: /content/evidencia_Q7_injecao_rollback.json

TESTE Q7 CONCLUIDO COM SUCESSO.


In [ ]:

import hashlib
import json
import subprocess
from pathlib import Path

# =====================================================
# EXPERIMENTO Q8 — SUPRESSAO DE GRUPOS PEQUENOS
# PAD-UFES-20 | PostgreSQL 16 | derm_v2
# =====================================================

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

def sql_run(sql, exigir_sucesso=True):
    r = subprocess.run(
        PSQL, input=sql,
        text=True, capture_output=True
    )
    if exigir_sucesso and r.returncode != 0:
        raise RuntimeError(r.stderr)
    return r

versao = sql_run("SHOW server_version;").stdout.strip()
assert versao.startswith("16."), versao

consulta_fonte = (
    "SELECT count(*) FROM derm_v2.fonte "
    "WHERE nome='PAD-UFES-20 metadata.csv' "
    f"AND versao='sha256:{SHA}';"
)

assert sql_run(consulta_fonte).stdout.strip() == "1"

# -----------------------------------------------------
# 1. CRIAR VISOES DE AUDITORIA E PAPEL RESTRITO
# -----------------------------------------------------

instalacao = f"""
BEGIN;

CREATE SCHEMA IF NOT EXISTS analitica_v2;

DO $role$
BEGIN
    IF NOT EXISTS (
        SELECT 1 FROM pg_roles
        WHERE rolname = 'derm_analista_pad'
    ) THEN
        CREATE ROLE derm_analista_pad
            NOLOGIN NOINHERIT;
    END IF;
END
$role$;

-- Classificacao POR PACIENTE derivada das imagens.
-- Se houver fototipos diferentes para o mesmo
-- paciente, a categoria sera 'discordante'.

CREATE OR REPLACE VIEW
analitica_v2.v_cubo_bruto_pad AS

WITH por_paciente AS (
    SELECT
        p.fonte_id,
        p.paciente_id,

        COUNT(DISTINCT o.valor_categoria)
            FILTER (
                WHERE o.estado = 'preenchido'
            ) AS tipos_distintos,

        MAX(o.valor_categoria)
            FILTER (
                WHERE o.estado = 'preenchido'
            ) AS unico_tipo,

        COUNT(o.observacao_id)
            FILTER (
                WHERE o.estado = 'ausente'
            ) AS ausencias

    FROM derm_v2.paciente p

    JOIN derm_v2.fonte f
        ON f.fonte_id = p.fonte_id

    LEFT JOIN derm_v2.lesao l
        ON l.fonte_id = p.fonte_id
       AND l.paciente_id = p.paciente_id

    LEFT JOIN derm_v2.imagem i
        ON i.fonte_id = l.fonte_id
       AND i.lesao_id = l.lesao_id

    LEFT JOIN derm_v2.observacao o
        ON o.fonte_id = i.fonte_id
       AND o.imagem_id = i.imagem_id
       AND o.atributo = 'fitzpatrick'
       AND o.unidade = 'imagem'

    WHERE f.nome = 'PAD-UFES-20 metadata.csv'
      AND f.versao = 'sha256:{SHA}'

    GROUP BY p.fonte_id, p.paciente_id
),

classificacao AS (
    SELECT
        fonte_id,
        paciente_id,

        CASE
            WHEN tipos_distintos > 1
                THEN 'discordante'

            WHEN tipos_distintos = 1
                 AND ausencias > 0
                THEN 'misto_preenchido_ausente'

            WHEN tipos_distintos = 1
                THEN unico_tipo

            WHEN ausencias > 0
                THEN 'ausente'

            ELSE 'sem_observacao'
        END::text AS fitzpatrick

    FROM por_paciente
)

SELECT
    fonte_id,

    'nao_disponivel_na_fonte'::text
        AS raca_ibge,

    fitzpatrick,

    'residencia_nao_documentada'::text
        AS regiao_residencia,

    'diagnostico_nao_estruturado'::text
        AS malignidade_documentada,

    COUNT(*)::bigint AS pacientes

FROM classificacao

GROUP BY fonte_id, fitzpatrick;


-- Visao protegida: somente grupos com n >= 5.
-- O limite nao pode ser alterado pelo analista.

CREATE OR REPLACE VIEW
analitica_v2.v_cubo_protegido_pad
WITH (security_barrier = true) AS

SELECT
    fonte_id,
    raca_ibge,
    fitzpatrick,
    regiao_residencia,
    malignidade_documentada,
    pacientes

FROM analitica_v2.v_cubo_bruto_pad

WHERE pacientes >= 5;


-- Papel analista: acesso apenas a visao protegida.

REVOKE ALL ON SCHEMA analitica_v2 FROM PUBLIC;

REVOKE ALL ON SCHEMA derm_v2
    FROM derm_analista_pad;

REVOKE ALL ON ALL TABLES IN SCHEMA derm_v2
    FROM derm_analista_pad;

REVOKE ALL ON analitica_v2.v_cubo_bruto_pad
    FROM PUBLIC, derm_analista_pad;

REVOKE ALL ON analitica_v2.v_cubo_protegido_pad
    FROM PUBLIC;

GRANT USAGE ON SCHEMA analitica_v2
    TO derm_analista_pad;

GRANT SELECT ON analitica_v2.v_cubo_protegido_pad
    TO derm_analista_pad;

COMMIT;
"""

sql_run(instalacao)

print("Visoes e papel restrito configurados.")

# -----------------------------------------------------
# 2. COMPARAR VISAO BRUTA E VISAO PROTEGIDA
# -----------------------------------------------------

verificacao = """
WITH esperado AS (
    SELECT *
    FROM analitica_v2.v_cubo_bruto_pad
    WHERE pacientes >= 5
),

diferencas AS (
    (
        SELECT * FROM esperado
        EXCEPT
        SELECT *
        FROM analitica_v2.v_cubo_protegido_pad
    )

    UNION ALL

    (
        SELECT *
        FROM analitica_v2.v_cubo_protegido_pad
        EXCEPT
        SELECT * FROM esperado
    )
)

SELECT json_build_object(

    'pacientes_agregados',
    (
        SELECT SUM(pacientes)
        FROM analitica_v2.v_cubo_bruto_pad
    ),

    'grupos_totais',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_bruto_pad
    ),

    'grupos_suprimidos',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_bruto_pad
        WHERE pacientes < 5
    ),

    'grupos_publicados',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_protegido_pad
    ),

    'grupos_pequenos_expostos',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_protegido_pad
        WHERE pacientes < 5
    ),

    'divergencias_na_regra_de_supressao',
    (
        SELECT COUNT(*) FROM diferencas
    ),

    'permissao_visao_protegida',
    has_table_privilege(
        'derm_analista_pad',
        'analitica_v2.v_cubo_protegido_pad',
        'SELECT'
    ),

    'permissao_visao_bruta',
    has_table_privilege(
        'derm_analista_pad',
        'analitica_v2.v_cubo_bruto_pad',
        'SELECT'
    ),

    'permissao_tabela_paciente',
    has_table_privilege(
        'derm_analista_pad',
        'derm_v2.paciente',
        'SELECT'
    ),

    'permissao_schema_derm',
    has_schema_privilege(
        'derm_analista_pad',
        'derm_v2',
        'USAGE'
    )
);
"""

resposta = sql_run(verificacao)

linhas = [
    linha
    for linha in resposta.stdout.splitlines()
    if linha.strip().startswith("{")
]

assert len(linhas) == 1

v = json.loads(linhas[0])

# -----------------------------------------------------
# 3. TESTAR ACESSO REAL COM SET ROLE
# -----------------------------------------------------

acesso_protegido = sql_run(
    """
    BEGIN;
    SET LOCAL ROLE derm_analista_pad;
    SELECT COUNT(*)
    FROM analitica_v2.v_cubo_protegido_pad;
    ROLLBACK;
    """,
    exigir_sucesso=False
)

acesso_bruto = sql_run(
    """
    BEGIN;
    SET LOCAL ROLE derm_analista_pad;
    SELECT COUNT(*)
    FROM analitica_v2.v_cubo_bruto_pad;
    ROLLBACK;
    """,
    exigir_sucesso=False
)

acesso_paciente = sql_run(
    """
    BEGIN;
    SET LOCAL ROLE derm_analista_pad;
    SELECT COUNT(*)
    FROM derm_v2.paciente;
    ROLLBACK;
    """,
    exigir_sucesso=False
)

# -----------------------------------------------------
# 4. CRITERIOS DE ACEITACAO
# -----------------------------------------------------

criterios = {
    "total_pacientes_preservado":
        v["pacientes_agregados"] == 1373,

    "nenhum_grupo_pequeno_exposto":
        v["grupos_pequenos_expostos"] == 0,

    "supressao_exata":
        v["divergencias_na_regra_de_supressao"] == 0,

    "papel_le_visao_protegida":
        acesso_protegido.returncode == 0
        and v["permissao_visao_protegida"],

    "papel_nao_le_visao_bruta":
        acesso_bruto.returncode != 0
        and not v["permissao_visao_bruta"],

    "papel_nao_le_pacientes":
        acesso_paciente.returncode != 0
        and not v["permissao_tabela_paciente"]
        and not v["permissao_schema_derm"]
}

# -----------------------------------------------------
# 5. RELATORIO
# -----------------------------------------------------

relatorio = {
    "experimento":
        "Q8 - supressao por k=5 e controle de acesso",

    "postgresql": versao,

    "sha256_csv": SHA,

    "unidade":
        "paciente; classificacao derivada de imagens",

    "dimensoes_sem_dados": [
        "raca/cor IBGE",
        "residencia",
        "diagnostico estruturado"
    ],

    "k": 5,

    "resultados": v,

    "criterios": criterios,

    "todos_atendidos": all(criterios.values()),

    "limites": (
        "Supressao simples nao garante anonimato "
        "contra inferencia por totais ou consultas "
        "complementares. Nao testa autenticacao "
        "externa nem politica geral multi-fonte."
    )
}

saida = Path(
    "/content/evidencia_Q8_supressao_pad.json"
)

saida.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    relatorio,
    ensure_ascii=False,
    indent=2
))

print("\nRelatorio salvo:", saida)

assert relatorio["todos_atendidos"], (
    "Um ou mais criterios falharam."
)

print("\nTESTE Q8 CONCLUIDO.")


Visoes e papel restrito configurados.
{
  "experimento": "Q8 - supressao por k=5 e controle de acesso",
  "postgresql": "16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "unidade": "paciente; classificacao derivada de imagens",
  "dimensoes_sem_dados": [
    "raca/cor IBGE",
    "residencia",
    "diagnostico estruturado"
  ],
  "k": 5,
  "resultados": {
    "pacientes_agregados": 1373,
    "grupos_totais": 7,
    "grupos_suprimidos": 1,
    "grupos_publicados": 6,
    "grupos_pequenos_expostos": 0,
    "divergencias_na_regra_de_supressao": 0,
    "permissao_visao_protegida": true,
    "permissao_visao_bruta": false,
    "permissao_tabela_paciente": false,
    "permissao_schema_derm": false
  },
  "criterios": {
    "total_pacientes_preservado": true,
    "nenhum_grupo_pequeno_exposto": true,
    "supressao_exata": true,
    "papel_le_visao_protegida": true,
    "papel_nao_le_visao_bruta": true,
    "papel_nao

In [ ]:

import json
import subprocess
from pathlib import Path

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

sql = """
BEGIN;

SET LOCAL ROLE derm_analista_pad;

SELECT COALESCE(SUM(pacientes), 0)::bigint
FROM analitica_v2.v_cubo_protegido_pad;

ROLLBACK;
"""

r = subprocess.run(
    PSQL,
    input=sql,
    text=True,
    capture_output=True
)

assert r.returncode == 0, r.stderr

linhas = [
    linha.strip()
    for linha in r.stdout.splitlines()
    if linha.strip()
]

assert len(linhas) == 1, r.stdout

total_publico = 1373
total_visivel = int(linhas[0])
total_inferido = total_publico - total_visivel

relatorio = {
    "experimento": "Q8 - inferencia complementar",
    "papel": "derm_analista_pad",
    "total_publicamente_conhecido": total_publico,
    "soma_dos_grupos_publicados": total_visivel,
    "quantidade_suprimida_inferida": total_inferido,
    "inferencia_de_grupo_pequeno": (
        0 < total_inferido < 5
    ),
    "observacao": (
        "O teste avalia inferencia por diferenca "
        "entre total publico e soma das contagens "
        "acessiveis ao papel restrito."
    )
}

print(json.dumps(
    relatorio,
    indent=2,
    ensure_ascii=False
))

Path(
    "/content/evidencia_Q8_inferencia.json"
).write_text(
    json.dumps(
        relatorio,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


{
  "experimento": "Q8 - inferencia complementar",
  "papel": "derm_analista_pad",
  "total_publicamente_conhecido": 1373,
  "soma_dos_grupos_publicados": 1372,
  "quantidade_suprimida_inferida": 1,
  "inferencia_de_grupo_pequeno": true,
  "observacao": "O teste avalia inferencia por diferenca entre total publico e soma das contagens acessiveis ao papel restrito."
}


368

In [ ]:

import json
import hashlib
import subprocess
from pathlib import Path

# ====================================================
# EXPERIMENTO Q8-B
# Supressao primaria + secundaria
# PostgreSQL 16 | PAD-UFES-20
# ====================================================

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

def executar(sql):
    r = subprocess.run(
        PSQL,
        input=sql,
        text=True,
        capture_output=True
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    return r.stdout.strip()

# A visao original permanece em v_cubo_bruto_pad.
# Somente a visao protegida sera modificada.

migracao = """
BEGIN;

CREATE OR REPLACE VIEW
analitica_v2.v_cubo_protegido_pad
WITH (security_barrier = true) AS

WITH classificados AS (
    SELECT
        b.*,

        COUNT(*) FILTER (
            WHERE pacientes < 5
        ) OVER (
            PARTITION BY fonte_id
        ) AS grupos_pequenos,

        ROW_NUMBER() OVER (
            PARTITION BY fonte_id
            ORDER BY pacientes DESC, fitzpatrick
        ) AS prioridade_secundaria

    FROM analitica_v2.v_cubo_bruto_pad b
)

SELECT
    fonte_id,
    raca_ibge,
    fitzpatrick,
    regiao_residencia,
    malignidade_documentada,
    pacientes

FROM classificados

WHERE pacientes >= 5

  AND NOT (
      grupos_pequenos > 0
      AND prioridade_secundaria = 1
  );

COMMIT;
"""

executar(migracao)

# Verificar a nova politica usando o SQL.

consulta = """
SELECT json_build_object(

    'grupos_totais',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_bruto_pad
    ),

    'grupos_primarios_pequenos',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_bruto_pad
        WHERE pacientes < 5
    ),

    'grupos_publicados',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_protegido_pad
    ),

    'grupos_suprimidos_total',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_bruto_pad
    ) -
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_protegido_pad
    ),

    'grupos_pequenos_expostos',
    (
        SELECT COUNT(*)
        FROM analitica_v2.v_cubo_protegido_pad
        WHERE pacientes < 5
    ),

    'total_pacientes',
    (
        SELECT SUM(pacientes)
        FROM analitica_v2.v_cubo_bruto_pad
    ),

    'soma_publicada',
    (
        SELECT COALESCE(SUM(pacientes), 0)
        FROM analitica_v2.v_cubo_protegido_pad
    )
);
"""

dados = json.loads(executar(consulta))

# Conferir acesso real do papel restrito.
acesso = """
BEGIN;
SET LOCAL ROLE derm_analista_pad;

SELECT COALESCE(SUM(pacientes), 0)
FROM analitica_v2.v_cubo_protegido_pad;

ROLLBACK;
"""

soma_analista = int(executar(acesso))

residual = (
    dados["total_pacientes"] -
    soma_analista
)

criterios = {
    "total_preservado":
        dados["total_pacientes"] == 1373,

    "grupo_pequeno_nao_exposto":
        dados["grupos_pequenos_expostos"] == 0,

    "supressao_secundaria_aplicada":
        dados["grupos_suprimidos_total"] >= 2,

    "soma_conferida_com_papel_restrito":
        soma_analista == dados["soma_publicada"],

    "diferenca_nao_revela_apenas_grupo_pequeno":
        residual >= 6
}

relatorio = {
    "experimento":
        "Q8-B - supressao complementar",

    "limiar_k": 5,

    "resultados": dados,

    "soma_visivel_ao_analista": soma_analista,

    "residual_inferivel": residual,

    "criterios": criterios,

    "todos_atendidos": all(criterios.values()),

    "limite": (
        "A supressao secundaria impede que a "
        "diferenca simples corresponda somente "
        "ao grupo primario. Nao garante "
        "anonimizacao contra informacoes auxiliares "
        "ou outras consultas."
    )
}

print(json.dumps(
    relatorio,
    ensure_ascii=False,
    indent=2
))

caminho = Path(
    "/content/evidencia_Q8_supressao_secundaria.json"
)

caminho.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nRelatorio salvo:", caminho)

assert relatorio["todos_atendidos"], (
    "Um ou mais criterios nao foram atendidos."
)

print("\nTESTE Q8-B CONCLUIDO.")


{
  "experimento": "Q8-B - supressao complementar",
  "limiar_k": 5,
  "resultados": {
    "grupos_totais": 7,
    "grupos_primarios_pequenos": 1,
    "grupos_publicados": 5,
    "grupos_suprimidos_total": 2,
    "grupos_pequenos_expostos": 0,
    "total_pacientes": 1373,
    "soma_publicada": 793
  },
  "soma_visivel_ao_analista": 793,
  "residual_inferivel": 580,
  "criterios": {
    "total_preservado": true,
    "grupo_pequeno_nao_exposto": true,
    "supressao_secundaria_aplicada": true,
    "soma_conferida_com_papel_restrito": true,
    "diferenca_nao_revela_apenas_grupo_pequeno": true
  },
  "todos_atendidos": true,
  "limite": "A supressao secundaria impede que a diferenca simples corresponda somente ao grupo primario. Nao garante anonimizacao contra informacoes auxiliares ou outras consultas."
}

Relatorio salvo: /content/evidencia_Q8_supressao_secundaria.json

TESTE Q8-B CONCLUIDO.


In [ ]:

import sys
import subprocess
import hashlib
import io
import json
import uuid
import struct
import zlib
from pathlib import Path

# SDK instalado somente no ambiente do Colab.
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-q", "minio==7.2.20"],
    check=True
)

from minio import Minio
from minio.error import S3Error

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

def sql(cmd):
    r = subprocess.run(
        PSQL, input=cmd,
        text=True, capture_output=True
    )
    if r.returncode:
        raise RuntimeError(r.stderr)
    return r.stdout.strip()

def json_sql(consulta):
    comando = (
        "SELECT COALESCE(json_agg(row_to_json(t)), "
        "'[]'::json) FROM ("
        + consulta.strip().rstrip(";")
        + ") t;"
    )
    return json.loads(sql(comando))

versao = sql("SHOW server_version;")
assert versao.startswith("16."), versao

# Identificar a PAD-UFES-20 sem alterar seus dados.
SHA_PAD = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

fontes = json_sql(f"""
SELECT fonte_id
FROM derm_v2.fonte
WHERE nome = 'PAD-UFES-20 metadata.csv'
  AND versao = 'sha256:{SHA_PAD}'
""")

assert len(fontes) == 1
fonte_pad = fontes[0]["fonte_id"]

def estado_pad():
    return json_sql(f"""
    SELECT
        (SELECT COUNT(*) FROM derm_v2.staging_registro
         WHERE fonte_id={fonte_pad}) AS staging,
        (SELECT COUNT(*) FROM derm_v2.paciente
         WHERE fonte_id={fonte_pad}) AS pacientes,
        (SELECT COUNT(*) FROM derm_v2.lesao
         WHERE fonte_id={fonte_pad}) AS lesoes,
        (SELECT COUNT(*) FROM derm_v2.imagem
         WHERE fonte_id={fonte_pad}) AS imagens,
        (SELECT COUNT(*) FROM derm_v2.observacao
         WHERE fonte_id={fonte_pad}) AS observacoes
    """)[0]

antes_pad = estado_pad()

assert antes_pad == {
    "staging": 2298,
    "pacientes": 1373,
    "lesoes": 1891,
    "imagens": 2298,
    "observacoes": 2298
}

# Gerador de PNG sem pessoas, EXIF ou informações clínicas.
def png_sintetico(semente):
    def bloco(tipo, corpo):
        material = tipo + corpo
        return (
            struct.pack("!I", len(corpo))
            + material
            + struct.pack(
                "!I", zlib.crc32(material) & 0xffffffff
            )
        )

    linhas = []
    for y in range(10):
        linha = bytearray([0])
        for x in range(12):
            linha.extend((
                (x * 19 + semente) % 256,
                (y * 23 + 3 * semente) % 256,
                (x * 7 + y * 11 + 5 * semente) % 256
            ))
        linhas.append(linha)

    ihdr = struct.pack(
        "!IIBBBBB", 12, 10, 8, 2, 0, 0, 0
    )

    return (
        b"\x89PNG\r\n\x1a\n"
        + bloco(b"IHDR", ihdr)
        + bloco(
            b"IDAT",
            zlib.compress(b"".join(linhas), 9)
        )
        + bloco(b"IEND", b"")
    )

# Servidor PUBLICO de demonstração do MinIO.
# Credenciais públicas documentadas no exemplo oficial.
minio = Minio(
    "play.min.io",
    access_key="Q3AM3UQ867SPQQA43P2F",
    secret_key="zuf+tfteSlswRu7BJ86wekitnifILbZam1KYY3TG",
    secure=True
)

rotulo = uuid.uuid4().hex
bucket = "derm-lab-" + rotulo[:18]
prefixo = "e7/" + rotulo + "/"

a = png_sintetico(31)
b = png_sintetico(57)

objetos = {
    prefixo + "A.png": a,
    prefixo + "B.png": b
}

fonte_lab = None
bucket_criado = False

def put(chave, dados):
    minio.put_object(
        bucket, chave,
        io.BytesIO(dados),
        len(dados),
        content_type="image/png"
    )

def auditar():
    referencias = json_sql(f"""
        SELECT
            fonte_id, imagem_id, bucket, chave_objeto,
            tamanho_bytes, sha256_hex
        FROM derm_v2.imagem
        WHERE fonte_id={fonte_lab}
    """)

    chaves_referenciadas = {
        r["chave_objeto"] for r in referencias
    }

    resultados = []

    for ref in referencias:
        status = "ok"
        tamanho_lido = 0
        sha = hashlib.sha256()

        try:
            resposta = minio.get_object(
                ref["bucket"], ref["chave_objeto"]
            )
            try:
                for pedaco in resposta.stream(32768):
                    if pedaco:
                        tamanho_lido += len(pedaco)
                        sha.update(pedaco)
            finally:
                resposta.close()
                resposta.release_conn()

            tamanho_errado = (
                tamanho_lido != ref["tamanho_bytes"]
            )
            hash_errado = (
                sha.hexdigest() != ref["sha256_hex"]
            )

            if tamanho_errado and hash_errado:
                status = "tamanho_e_sha256_divergentes"
            elif tamanho_errado:
                status = "tamanho_divergente"
            elif hash_errado:
                status = "sha256_divergente"

        except S3Error as erro:
            if erro.code in (
                "NoSuchKey", "NoSuchBucket",
                "NotFound", "NoSuchObject"
            ):
                status = "objeto_ausente"
            else:
                raise

        resultados.append({
            "direcao": "banco_para_objeto",
            "imagem_id": ref["imagem_id"],
            "chave": ref["chave_objeto"],
            "status": status
        })

    for obj in minio.list_objects(
        bucket, prefix=prefixo, recursive=True
    ):
        if obj.object_name not in chaves_referenciadas:
            resultados.append({
                "direcao": "objeto_para_banco",
                "imagem_id": None,
                "chave": obj.object_name,
                "status": "orfao"
            })

    return resultados

def detectado(casos, status, chave=None, imagem_id=None):
    return any(
        c["status"] == status
        and (chave is None or c["chave"] == chave)
        and (
            imagem_id is None
            or c["imagem_id"] == imagem_id
        )
        for c in casos
    )

relatorio = {
    "experimento": "Integridade PostgreSQL 16 - MinIO",
    "dados": "Somente PNG sinteticos",
    "postgresql": versao,
    "minio_sdk": "7.2.20",
    "endpoint": "play.min.io",
    "resultados": {},
    "criterios": {},
    "pad_antes": antes_pad
}

try:
    minio.make_bucket(bucket)
    bucket_criado = True

    fonte_lab = int(sql(f"""
        INSERT INTO derm_v2.fonte(nome, versao)
        VALUES (
            'LAB_MINIO_{rotulo}',
            'fixture-sintetica'
        )
        RETURNING fonte_id;
    """))

    chaves = list(objetos)

    for i, chave in enumerate(chaves):
        dados_png = objetos[chave]
        sha = hashlib.sha256(dados_png).hexdigest()

        sql(f"""
            INSERT INTO derm_v2.imagem (
                fonte_id,
                identificador_origem,
                modalidade,
                bucket,
                chave_objeto,
                tamanho_bytes,
                sha256_hex
            )
            VALUES (
                {fonte_lab},
                'fixture-{i}',
                'clinica',
                '{bucket}',
                '{chave}',
                {len(dados_png)},
                '{sha}'
            );
        """)

        put(chave, dados_png)

    linhas = json_sql(f"""
        SELECT imagem_id, chave_objeto
        FROM derm_v2.imagem
        WHERE fonte_id={fonte_lab}
        ORDER BY identificador_origem
    """)

    id_a = linhas[0]["imagem_id"]
    chave_a, chave_b = chaves

    # TESTE 0 — Arquivos íntegros
    base = auditar()

    assert len(base) == 2
    assert all(c["status"] == "ok" for c in base)

    relatorio["criterios"]["baseline_ok"] = True
    relatorio["resultados"]["baseline"] = base

    # TESTE 1 — Objeto apagado
    try:
        minio.remove_object(bucket, chave_a)
        casos = auditar()

        relatorio["criterios"]["objeto_ausente"] = (
            detectado(
                casos, "objeto_ausente", chave_a, id_a
            )
        )
        relatorio["resultados"]["objeto_ausente"] = casos
    finally:
        put(chave_a, a)

    # TESTE 2 — Bytes corrompidos, mesmo tamanho
    try:
        corrompido = bytearray(a)
        corrompido[-15] ^= 1
        put(chave_a, bytes(corrompido))

        casos = auditar()
        relatorio["criterios"]["hash_divergente"] = (
            detectado(
                casos, "sha256_divergente",
                chave_a, id_a
            )
        )
        relatorio["resultados"]["hash_divergente"] = casos
    finally:
        put(chave_a, a)

    # TESTE 3 — Arquivo truncado
    try:
        put(chave_a, a[:len(a) // 2])
        casos = auditar()

        relatorio["criterios"]["truncamento"] = (
            detectado(
                casos, "tamanho_e_sha256_divergentes",
                chave_a, id_a
            )
        )
        relatorio["resultados"]["truncamento"] = casos
    finally:
        put(chave_a, a)

    # TESTE 4 — Objeto sem referência no banco
    chave_orfa = prefixo + "orfao.png"

    try:
        put(chave_orfa, png_sintetico(101))
        casos = auditar()

        relatorio["criterios"]["orfao"] = (
            detectado(
                casos, "orfao", chave_orfa
            )
        )
        relatorio["resultados"]["orfao"] = casos
    finally:
        minio.remove_object(bucket, chave_orfa)

    # TESTE 5 — Referência apontando para objeto errado
    esperado = (
        "sha256_divergente"
        if len(a) == len(b)
        else "tamanho_e_sha256_divergentes"
    )

    try:
        sql(f"""
            UPDATE derm_v2.imagem
            SET chave_objeto='{chave_b}'
            WHERE imagem_id={id_a}
              AND fonte_id={fonte_lab};
        """)

        casos = auditar()
        relatorio["criterios"]["troca_chave"] = (
            detectado(casos, esperado, chave_b, id_a)
            and detectado(casos, "orfao", chave_a)
        )
        relatorio["resultados"]["troca_chave"] = casos
    finally:
        sql(f"""
            UPDATE derm_v2.imagem
            SET chave_objeto='{chave_a}'
            WHERE imagem_id={id_a}
              AND fonte_id={fonte_lab};
        """)

    # Confirmar que a fixture voltou ao estado íntegro.
    final = auditar()

    relatorio["criterios"]["baseline_restaurada"] = (
        len(final) == 2
        and all(c["status"] == "ok" for c in final)
    )

    relatorio["resultados"]["baseline_restaurada"] = final

finally:
    # Limpeza isolada: não remove a PAD-UFES-20.
    erros_limpeza = []

    if fonte_lab is not None:
        try:
            sql(f"""
                DELETE FROM derm_v2.imagem
                WHERE fonte_id={fonte_lab};

                DELETE FROM derm_v2.fonte
                WHERE fonte_id={fonte_lab};
            """)
        except Exception as erro:
            erros_limpeza.append(
                "postgresql: " + str(erro)
            )

    if bucket_criado:
        try:
            for obj in minio.list_objects(
                bucket,
                prefix=prefixo,
                recursive=True
            ):
                minio.remove_object(
                    bucket, obj.object_name
                )

            minio.remove_bucket(bucket)

        except Exception as erro:
            erros_limpeza.append(
                "minio: " + str(erro)
            )

    relatorio["pad_depois"] = estado_pad()

    relatorio["criterios"]["pad_inalterada"] = (
        relatorio["pad_depois"] == antes_pad
    )

    relatorio["limpeza_ok"] = not erros_limpeza
    relatorio["erros_limpeza"] = erros_limpeza

    relatorio["criterios"]["limpeza_ok"] = (
        not erros_limpeza
    )

    relatorio["todos_atendidos"] = all(
        relatorio["criterios"].values()
    )

    caminho = Path(
        "/content/evidencia_postgresql_minio_play.json"
    )

    caminho.write_text(
        json.dumps(
            relatorio,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    print(json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ))

    print("\nRelatorio salvo:", caminho)

assert relatorio["todos_atendidos"], (
    "Uma ou mais verificacoes falharam."
)

print("\nEXPERIMENTO CONCLUIDO.")


{
  "experimento": "Integridade PostgreSQL 16 - MinIO",
  "dados": "Somente PNG sinteticos",
  "postgresql": "16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)",
  "minio_sdk": "7.2.20",
  "endpoint": "play.min.io",
  "resultados": {
    "baseline": [
      {
        "direcao": "banco_para_objeto",
        "imagem_id": 2301,
        "chave": "e7/6bd571c273fe418198b971f42708f60e/A.png",
        "status": "ok"
      },
      {
        "direcao": "banco_para_objeto",
        "imagem_id": 2302,
        "chave": "e7/6bd571c273fe418198b971f42708f60e/B.png",
        "status": "ok"
      }
    ],
    "objeto_ausente": [
      {
        "direcao": "banco_para_objeto",
        "imagem_id": 2301,
        "chave": "e7/6bd571c273fe418198b971f42708f60e/A.png",
        "status": "objeto_ausente"
      },
      {
        "direcao": "banco_para_objeto",
        "imagem_id": 2302,
        "chave": "e7/6bd571c273fe418198b971f42708f60e/B.png",
        "status": "ok"
      }
    ],
    "hash_divergente": [
      {
   

In [16]:

import hashlib
import json
import uuid
from pathlib import Path

# =====================================================
# EXPERIMENTO E3 — 25 CENARIOS DE FALHA
# PostgreSQL 16 + MinIO real
# =====================================================

# Reutiliza as funções criadas no experimento anterior.
necessarios = [
    "minio", "png_sintetico", "sql", "json_sql",
    "auditar", "put", "estado_pad", "conteudo"
]

for nome in necessarios:
    assert nome in globals(), (
        f"Reexecute a célula PostgreSQL-MinIO anterior: {nome}"
    )

SHA_PAD = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA_PAD

pad_antes = estado_pad()

assert pad_antes == {
    "staging": 2298,
    "pacientes": 1373,
    "lesoes": 1891,
    "imagens": 2298,
    "observacoes": 2298
}

# Novo bucket e nova fonte exclusivamente sintéticos.
# As funções put() e auditar() utilizam estas variáveis.

bucket = "derm-lab-" + uuid.uuid4().hex[:18]
prefixo = "e8/" + uuid.uuid4().hex + "/"

fonte_lab = None
bucket_criado = False

chaves = [
    prefixo + f"cenario-{i}.png"
    for i in range(6)
]

conteudos = [
    png_sintetico(20 + 17 * i)
    for i in range(6)
]

ids = {}

relatorio = {
    "experimento": "E3 - 5 cenarios por tipo de falha",
    "dados": "Somente PNG sinteticos",
    "postgresql": sql("SHOW server_version;"),
    "endpoint": "play.min.io",
    "sha256_pad": SHA_PAD,
    "casos": [],
    "baselines_verificadas": 0,
    "pad_antes": pad_antes
}

# =====================================================
# FUNCOES DE VERIFICACAO
# =====================================================

def verificar_base_limpa():
    casos = auditar()

    assert len(casos) == 6, (
        f"Baseline incompleta: {casos}"
    )

    assert all(
        c["status"] == "ok"
        for c in casos
    ), f"Alerta inesperado na baseline: {casos}"

    relatorio["baselines_verificadas"] += 1
    return True


def registrar(categoria, indice, casos):
    falhas = [
        c for c in casos
        if c["status"] != "ok"
    ]

    imagem_id = ids[indice]
    chave = chaves[indice]

    if categoria == "objeto_ausente":
        passou = (
            len(falhas) == 1
            and falhas[0]["imagem_id"] == imagem_id
            and falhas[0]["status"] == "objeto_ausente"
        )

    elif categoria == "sha256_divergente":
        passou = (
            len(falhas) == 1
            and falhas[0]["imagem_id"] == imagem_id
            and falhas[0]["status"] == "sha256_divergente"
        )

    elif categoria == "truncamento":
        passou = (
            len(falhas) == 1
            and falhas[0]["imagem_id"] == imagem_id
            and falhas[0]["status"]
                == "tamanho_e_sha256_divergentes"
        )

    elif categoria == "orfao":
        esperado = prefixo + f"orfao-{indice}.png"

        passou = (
            len(falhas) == 1
            and falhas[0]["imagem_id"] is None
            and falhas[0]["chave"] == esperado
            and falhas[0]["status"] == "orfao"
        )

    elif categoria == "troca_chave":
        divergencias = [
            c for c in falhas
            if c["imagem_id"] == imagem_id
            and c["chave"] == chaves[5]
            and c["status"] in (
                "sha256_divergente",
                "tamanho_e_sha256_divergentes",
                "tamanho_divergente"
            )
        ]

        orfaos = [
            c for c in falhas
            if c["imagem_id"] is None
            and c["chave"] == chave
            and c["status"] == "orfao"
        ]

        passou = (
            len(falhas) == 2
            and len(divergencias) == 1
            and len(orfaos) == 1
        )

    else:
        raise ValueError(categoria)

    relatorio["casos"].append({
        "tipo": categoria,
        "replica": indice + 1,
        "detectado_corretamente": bool(passou),
        "anomalias": falhas
    })

    assert passou, (
        f"Falha em {categoria}, réplica {indice + 1}: "
        f"{casos}"
    )


# =====================================================
# INSTALACAO DA FIXTURE
# =====================================================

try:
    minio.make_bucket(bucket)
    bucket_criado = True

    fonte_lab = int(sql(f"""
        INSERT INTO derm_v2.fonte(nome, versao)
        VALUES (
            'LAB_E3_{uuid.uuid4().hex}',
            'sintetico'
        )
        RETURNING fonte_id;
    """))

    for i, (chave, dados) in enumerate(
        zip(chaves, conteudos)
    ):
        h = hashlib.sha256(dados).hexdigest()

        imagem_id = int(sql(f"""
            INSERT INTO derm_v2.imagem (
                fonte_id,
                identificador_origem,
                modalidade,
                bucket,
                chave_objeto,
                tamanho_bytes,
                sha256_hex
            )
            VALUES (
                {fonte_lab},
                'e3-{i}',
                'clinica',
                '{bucket}',
                '{chave}',
                {len(dados)},
                '{h}'
            )
            RETURNING imagem_id;
        """))

        ids[i] = imagem_id
        put(chave, dados)

    verificar_base_limpa()
    relatorio["baseline_inicial_ok"] = True

    # =================================================
    # FALHA 1 — OBJETO AUSENTE (5 CENARIOS)
    # =================================================

    for i in range(5):
        verificar_base_limpa()

        try:
            minio.remove_object(bucket, chaves[i])

            registrar(
                "objeto_ausente",
                i,
                auditar()
            )
        finally:
            put(chaves[i], conteudos[i])

        verificar_base_limpa()

    print("Objeto ausente: 5 cenarios executados.")

    # =================================================
    # FALHA 2 — SHA-256 DIVERGENTE (5 CENARIOS)
    # =================================================

    for i in range(5):
        verificar_base_limpa()

        try:
            alterado = bytearray(conteudos[i])
            alterado[-15] ^= 1

            put(chaves[i], bytes(alterado))

            registrar(
                "sha256_divergente",
                i,
                auditar()
            )
        finally:
            put(chaves[i], conteudos[i])

        verificar_base_limpa()

    print("SHA-256 divergente: 5 cenarios executados.")

    # =================================================
    # FALHA 3 — TRUNCAMENTO (5 CENARIOS)
    # =================================================

    for i in range(5):
        verificar_base_limpa()

        try:
            put(
                chaves[i],
                conteudos[i][:len(conteudos[i]) // 2]
            )

            registrar(
                "truncamento",
                i,
                auditar()
            )
        finally:
            put(chaves[i], conteudos[i])

        verificar_base_limpa()

    print("Truncamento: 5 cenarios executados.")

    # =================================================
    # FALHA 4 — OBJETO ORFAO (5 CENARIOS)
    # =================================================

    for i in range(5):
        verificar_base_limpa()

        chave_orfa = prefixo + f"orfao-{i}.png"

        try:
            put(
                chave_orfa,
                png_sintetico(120 + i)
            )

            registrar(
                "orfao",
                i,
                auditar()
            )
        finally:
            minio.remove_object(bucket, chave_orfa)

        verificar_base_limpa()

    print("Objeto orfao: 5 cenarios executados.")

    # =================================================
    # FALHA 5 — REFERENCIA INCORRETA (5 CENARIOS)
    # =================================================

    for i in range(5):
        verificar_base_limpa()

        try:
            sql(f"""
                UPDATE derm_v2.imagem
                SET chave_objeto = '{chaves[5]}'
                WHERE imagem_id = {ids[i]}
                  AND fonte_id = {fonte_lab};
            """)

            registrar(
                "troca_chave",
                i,
                auditar()
            )
        finally:
            sql(f"""
                UPDATE derm_v2.imagem
                SET chave_objeto = '{chaves[i]}'
                WHERE imagem_id = {ids[i]}
                  AND fonte_id = {fonte_lab};
            """)

        verificar_base_limpa()

    print("Referencia incorreta: 5 cenarios executados.")

    relatorio["baseline_final_ok"] = (
        verificar_base_limpa()
    )

# =====================================================
# LIMPEZA E RELATORIO
# =====================================================

finally:
    erros_limpeza = []

    if fonte_lab is not None:
        try:
            sql(f"""
                DELETE FROM derm_v2.imagem
                WHERE fonte_id = {fonte_lab};

                DELETE FROM derm_v2.fonte
                WHERE fonte_id = {fonte_lab};
            """)
        except Exception as e:
            erros_limpeza.append(
                "postgresql: " + str(e)
            )

    if bucket_criado:
        try:
            for obj in minio.list_objects(
                bucket,
                prefix=prefixo,
                recursive=True
            ):
                minio.remove_object(
                    bucket,
                    obj.object_name
                )

            minio.remove_bucket(bucket)

        except Exception as e:
            erros_limpeza.append(
                "minio: " + str(e)
            )

    relatorio["pad_depois"] = estado_pad()

    relatorio["pad_inalterada"] = (
        relatorio["pad_depois"] == pad_antes
    )

    relatorio["limpeza_ok"] = not erros_limpeza
    relatorio["erros_limpeza"] = erros_limpeza

    relatorio["casos_executados"] = len(
        relatorio["casos"]
    )

    relatorio["casos_detectados"] = sum(
        c["detectado_corretamente"]
        for c in relatorio["casos"]
    )

    tipos = [
        "objeto_ausente",
        "sha256_divergente",
        "truncamento",
        "orfao",
        "troca_chave"
    ]

    relatorio["casos_por_tipo"] = {
        tipo: sum(
            c["tipo"] == tipo
            for c in relatorio["casos"]
        )
        for tipo in tipos
    }

    relatorio["todos_atendidos"] = all([
        relatorio["casos_executados"] == 25,
        relatorio["casos_detectados"] == 25,
        relatorio["baselines_verificadas"] == 52,
        relatorio.get("baseline_inicial_ok") is True,
        relatorio.get("baseline_final_ok") is True,
        relatorio["pad_inalterada"],
        relatorio["limpeza_ok"]
    ])

    destino = Path(
        "/content/evidencia_E3_minio_25_cenarios.json"
    )

    destino.write_text(
        json.dumps(
            relatorio,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    resumo = {
        chave: relatorio[chave]
        for chave in [
            "experimento",
            "casos_executados",
            "casos_detectados",
            "casos_por_tipo",
            "baselines_verificadas",
            "pad_inalterada",
            "limpeza_ok",
            "todos_atendidos"
        ]
    }

    print("\nRESULTADO DA BATERIA E3")
    print(json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2
    ))

    print("\nRelatorio completo:", destino)

assert relatorio["todos_atendidos"], (
    "A bateria nao atendeu a todos os criterios. "
    "Examine o relatorio."
)

print("\nBATERIA E3 CONCLUIDA.")


Objeto ausente: 5 cenarios executados.
SHA-256 divergente: 5 cenarios executados.
Truncamento: 5 cenarios executados.
Objeto orfao: 5 cenarios executados.
Referencia incorreta: 5 cenarios executados.

RESULTADO DA BATERIA E3
{
  "experimento": "E3 - 5 cenarios por tipo de falha",
  "casos_executados": 25,
  "casos_detectados": 25,
  "casos_por_tipo": {
    "objeto_ausente": 5,
    "sha256_divergente": 5,
    "truncamento": 5,
    "orfao": 5,
    "troca_chave": 5
  },
  "baselines_verificadas": 52,
  "pad_inalterada": true,
  "limpeza_ok": true,
  "todos_atendidos": true
}

Relatorio completo: /content/evidencia_E3_minio_25_cenarios.json

BATERIA E3 CONCLUIDA.


In [17]:
# E5 — Piloto de desempenho (Q1, Q2, Q4, Q6)
# Execução: Colab com PostgreSQL 16 / derm_v2 / PAD-UFES-20 já carregados.
import hashlib
import json
import os
import platform
import secrets
import statistics
import subprocess
import sys
import time
import uuid
from pathlib import Path

import pandas as pd

SHA = '14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527'
assert hashlib.sha256(conteudo).hexdigest() == SHA, 'CSV incorreto'
assert len(df) == 2298

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psycopg[binary]==3.2.3'], check=True)
import psycopg

ADMIN = [
    'runuser', '-u', 'postgres', '--', 'psql',
    '-X', '-q', '-t', '-A', '-v', 'ON_ERROR_STOP=1', '-d', 'postgres'
]

def executar_admin(comando):
    res = subprocess.run(ADMIN, input=comando, capture_output=True, text=True)
    if res.returncode:
        raise RuntimeError(res.stderr)
    return res.stdout.strip()

pg_versao = executar_admin('SHOW server_version;')
assert pg_versao.startswith('16.'), pg_versao
id_fonte = int(executar_admin(
    "SELECT fonte_id FROM derm_v2.fonte "
    "WHERE nome='PAD-UFES-20 metadata.csv' AND versao='sha256:" + SHA + "';"
))

# Credencial efêmera só para localhost, com SELECT nas quatro tabelas consultadas.
papel = 'derm_bench_' + uuid.uuid4().hex[:12]
senha = secrets.token_urlsafe(24)
con = None
papel_criado = False

SQL = {
 'Q1_cobertura_fitz': """
 WITH f AS (SELECT %s::bigint AS id)
 SELECT
 (SELECT COUNT(*) FROM derm_v2.imagem WHERE fonte_id=(SELECT id FROM f)) AS imagens,
 (SELECT COUNT(*) FROM derm_v2.observacao WHERE fonte_id=(SELECT id FROM f)
    AND atributo='fitzpatrick' AND unidade='imagem' AND estado='preenchido') AS preenchidas,
 (SELECT COUNT(*) FROM derm_v2.observacao WHERE fonte_id=(SELECT id FROM f)
    AND atributo='fitzpatrick' AND unidade='imagem' AND estado='ausente') AS ausentes
 """,
 'Q2_distribuicao_fitz': """
 SELECT CASE WHEN estado='preenchido' THEN valor_categoria
             ELSE '__' || estado::text || '__' END AS classe,
        COUNT(*) AS n
 FROM derm_v2.observacao
 WHERE fonte_id=%s AND atributo='fitzpatrick' AND unidade='imagem'
 GROUP BY 1 ORDER BY 1
 """,
 'Q4_granularidade': """
 WITH f AS (SELECT %s::bigint AS id)
 SELECT
 (SELECT COUNT(*) FROM derm_v2.paciente WHERE fonte_id=(SELECT id FROM f)) AS pacientes,
 (SELECT COUNT(*) FROM derm_v2.lesao WHERE fonte_id=(SELECT id FROM f)) AS lesoes,
 (SELECT COUNT(*) FROM derm_v2.imagem WHERE fonte_id=(SELECT id FROM f)) AS imagens
 """,
 'Q6_vinculos': """
 WITH f AS (SELECT %s::bigint AS id)
 SELECT
 (SELECT COUNT(*) FROM derm_v2.imagem WHERE fonte_id=(SELECT id FROM f)
     AND lesao_id IS NULL) AS imagens_sem_lesao,
 (SELECT COUNT(*) FROM derm_v2.lesao WHERE fonte_id=(SELECT id FROM f)
     AND paciente_id IS NULL) AS lesoes_sem_paciente
 """
}

MAP = {'': '__ausente__', '1.0':'I','2.0':'II','3.0':'III',
       '4.0':'IV','5.0':'V','6.0':'VI'}

def pandas_query(nome):
    if nome == 'Q1_cobertura_fitz':
        valores = df['fitspatrick'].fillna('').astype(str).str.strip()
        return {'imagens': int(df['img_id'].nunique()),
                'preenchidas': int(valores.ne('').sum()),
                'ausentes': int(valores.eq('').sum())}
    if nome == 'Q2_distribuicao_fitz':
        valores = df['fitspatrick'].fillna('').astype(str).str.strip()
        classificacoes = valores.map(MAP)
        assert classificacoes.notna().all(), 'Fitzpatrick inesperado no CSV'
        return {str(k):int(v) for k,v in classificacoes.value_counts().items()}
    if nome == 'Q4_granularidade':
        return {'pacientes': int(df['patient_id'].nunique()),
                'lesoes': int(df[['patient_id','lesion_id']].drop_duplicates().shape[0]),
                'imagens': int(df['img_id'].nunique())}
    if nome == 'Q6_vinculos':
        return {'imagens_sem_lesao': int(df['lesion_id'].fillna('').astype(str).str.strip().eq('').sum()),
                'lesoes_sem_paciente': int(df['patient_id'].fillna('').astype(str).str.strip().eq('').sum())}
    raise KeyError(nome)

def sql_query(nome):
    with con.cursor() as cur:
        cur.execute(SQL[nome], (id_fonte,))
        linhas = cur.fetchall()
        colunas = [x.name for x in cur.description]
    if nome == 'Q2_distribuicao_fitz':
        return {str(x[0]):int(x[1]) for x in linhas}
    assert len(linhas)==1, nome
    return dict(zip(colunas, [int(n) for n in linhas[0]]))

def quantil(valores, p):
    ordem = sorted(valores)
    pos = (len(ordem)-1)*p
    i = int(pos)
    j = min(i+1, len(ordem)-1)
    return ordem[i] + (ordem[j] - ordem[i])*(pos-i)

def resumo_tempos(valores):
    return {'n':len(valores), 'mediana_ms':statistics.median(valores),
            'q1_ms':quantil(valores,.25), 'q3_ms':quantil(valores,.75),
            'iiq_ms':quantil(valores,.75)-quantil(valores,.25),
            'max_ms':max(valores)}

def medir(func, nome):
    inicio = time.perf_counter_ns()
    func(nome)
    return (time.perf_counter_ns()-inicio)/1_000_000

try:
    executar_admin(f"CREATE ROLE {papel} LOGIN NOSUPERUSER NOCREATEDB NOCREATEROLE PASSWORD '{senha}';")
    papel_criado = True
    executar_admin(f"GRANT USAGE ON SCHEMA derm_v2 TO {papel}; "
                   f"GRANT SELECT ON derm_v2.paciente, derm_v2.lesao, "
                   f"derm_v2.imagem, derm_v2.observacao TO {papel};")
    con = psycopg.connect(host='127.0.0.1', port=5432, dbname='postgres',
                          user=papel, password=senha, autocommit=True,
                          connect_timeout=10)
    with con.cursor() as cur:
        cur.execute("SET default_transaction_read_only = on;")
        cur.execute("SET statement_timeout = '60s';")
    print('Conectado ao PostgreSQL:', pg_versao)
    configuracao_pg = {chave: executar_admin('SHOW ' + chave + ';') for chave in
                       ('shared_buffers','work_mem','effective_cache_size','max_parallel_workers_per_gather')}
    indices = executar_admin("SELECT indexname FROM pg_indexes WHERE schemaname='derm_v2' "
                             "AND tablename IN ('imagem','lesao','paciente','observacao') ORDER BY indexname;").splitlines()
    saida = {
      'experimento':'E5 — piloto comparativo, consultas Q1/Q2/Q4/Q6',
      'sha256_csv':SHA, 'fonte_id':id_fonte,
      'ambiente':{'postgresql':pg_versao, 'python':platform.python_version(),
                  'pandas':pd.__version__, 'psycopg':psycopg.__version__,
                  'sistema':platform.platform(), 'cpu_disponiveis':os.cpu_count(),
                  'csv_linhas':len(df), 'csv_colunas':len(df.columns),
                  'csv_bytes':len(conteudo),
                  'memoria_total_kib':int(next(l.split()[1] for l in Path('/proc/meminfo').read_text().splitlines() if l.startswith('MemTotal:')))},
      'postgresql_config':configuracao_pg, 'indices':indices,
      'configuracao':{'aquecimentos_por_consulta_e_metodo':3,
                      'repeticoes_por_consulta_e_metodo':20,
                      'ordem':'alternada SQL/pandas por repeticao; conexao SQL persistente',
                      'tempos':'somente consulta sobre dados ja carregados, ms'},
      'consultas':{}, 'criterios':{}
    }
    for nome in SQL:
        sql_r = sql_query(nome)
        pandas_r = pandas_query(nome)
        if sql_r != pandas_r:
            raise AssertionError(f'Resultados divergentes em {nome}: SQL={sql_r}, pandas={pandas_r}')
        for _ in range(3):
            medir(sql_query,nome)
            medir(pandas_query,nome)
        tempos_sql, tempos_pd = [], []
        for i in range(20):
            if i%2 == 0:
                tempos_sql.append(medir(sql_query,nome))
                tempos_pd.append(medir(pandas_query,nome))
            else:
                tempos_pd.append(medir(pandas_query,nome))
                tempos_sql.append(medir(sql_query,nome))
        r_sql, r_pd = resumo_tempos(tempos_sql), resumo_tempos(tempos_pd)
        razao = (r_sql['mediana_ms']/r_pd['mediana_ms']) if r_pd['mediana_ms'] > 0 else None
        with con.cursor() as cur:
            cur.execute('EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) ' + SQL[nome], (id_fonte,))
            plano = cur.fetchone()[0]
        saida['consultas'][nome] = {
            'explain_analyze_buffers':plano,
            'resultado_conferido':sql_r, 'concordancia_sql_pandas':True,
            'sql':r_sql, 'pandas':r_pd, 'razao_mediana_sql_sobre_pandas':razao,
            'meta_razao_ate_10':razao is not None and razao<=10,
            'sem_tempo_acima_60s':max(tempos_sql+tempos_pd)<=60000,
            'tempos_sql_ms':tempos_sql, 'tempos_pandas_ms':tempos_pd,
        }
        print(nome, '| SQL:',round(r_sql['mediana_ms'],4),'ms | pandas:',
              round(r_pd['mediana_ms'],4),'ms | SQL/pandas:',
              None if razao is None else round(razao,3))
    saida['criterios'] = {
        'quatro_consultas_conferidas':len(saida['consultas'])==4 and all(
            v['concordancia_sql_pandas'] for v in saida['consultas'].values()),
        'vinte_repeticoes_validas_por_metodo':all(v['sql']['n']==20 and v['pandas']['n']==20
                                           for v in saida['consultas'].values()),
        'nenhuma_execucao_acima_60s':all(v['sem_tempo_acima_60s'] for v in saida['consultas'].values()),
        'meta_exploratoria_razao_ate_10_todas':all(v['meta_razao_ate_10'] for v in saida['consultas'].values()),
    }
    saida['limites'] = ('Piloto de 4 consultas; protocolo completo E5 requer Q1-Q8. '
                      'Medição local com caches aquecidos, consultas SQL via conexão persistente '
                      'e pandas já em memória; não estima o custo de ingestão nem desempenho geral. '
                      'Resultado de Q6 compara apenas a ausência de identificadores/vínculos no recorte carregado.')
    caminho = Path('/content/evidencia_E5_piloto_4_consultas.json')
    caminho.write_text(json.dumps(saida,ensure_ascii=False,indent=2),encoding='utf-8')
    print('\nCRITERIOS:',json.dumps(saida['criterios'],indent=2,ensure_ascii=False))
    print('Evidência salva em:',caminho)
finally:
    if con is not None:
        con.close()
    if papel_criado:
        # Não altera os dados nem os papéis de pesquisas anteriores.
        executar_admin(f"REVOKE SELECT ON derm_v2.paciente, derm_v2.lesao, "
                       f"derm_v2.imagem, derm_v2.observacao FROM {papel}; "
                       f"REVOKE USAGE ON SCHEMA derm_v2 FROM {papel}; "
                       f"DROP ROLE {papel};")


Conectado ao PostgreSQL: 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)
Q1_cobertura_fitz | SQL: 1.3803 ms | pandas: 2.7262 ms | SQL/pandas: 0.506
Q2_distribuicao_fitz | SQL: 1.9858 ms | pandas: 3.1923 ms | SQL/pandas: 0.622
Q4_granularidade | SQL: 1.9158 ms | pandas: 3.6566 ms | SQL/pandas: 0.524
Q6_vinculos | SQL: 1.3341 ms | pandas: 2.8295 ms | SQL/pandas: 0.472

CRITERIOS: {
  "quatro_consultas_conferidas": true,
  "vinte_repeticoes_validas_por_metodo": true,
  "nenhuma_execucao_acima_60s": true,
  "meta_exploratoria_razao_ate_10_todas": true
}
Evidência salva em: /content/evidencia_E5_piloto_4_consultas.json


In [18]:
# E5 COMPLEMENTAR — Q3, Q5, Q7 e Q8 (sem ZIP)
# Pre-requisitos: Google Colab com CSV original carregado em `conteudo`/`df`,
# PostgreSQL 16, `derm_v2` e visao Q8-B instalada.
import hashlib, json, os, platform, secrets, statistics, subprocess, sys, time, uuid
from collections import defaultdict, Counter
from pathlib import Path
import pandas as pd

SHA = "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527"
assert hashlib.sha256(conteudo).hexdigest() == SHA, "CSV diferente do original"
assert len(df) == 2298, "Quantidade de linhas inesperada"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psycopg[binary]==3.2.3'], check=True)
import psycopg

ADMIN = ['runuser', '-u', 'postgres', '--', 'psql', '-X', '-q', '-t', '-A', '-v', 'ON_ERROR_STOP=1', '-d', 'postgres']
def admin(q):
    p = subprocess.run(ADMIN, input=q, capture_output=True, text=True)
    if p.returncode: raise RuntimeError(p.stderr)
    return p.stdout.strip()

versao = admin('SHOW server_version;')
assert versao.startswith('16.'), versao
ids_fonte = admin("SELECT fonte_id FROM derm_v2.fonte WHERE nome='PAD-UFES-20 metadata.csv' AND versao='sha256:"+SHA+"';").splitlines()
assert len(ids_fonte) == 1
fonte = int(ids_fonte[0])
# Garantir que a Q8-B (com supressao secundaria) existe.
q8_def = admin("SELECT pg_get_viewdef('analitica_v2.v_cubo_protegido_pad'::regclass, true);")
assert 'prioridade_secundaria' in q8_def or 'row_number()' in q8_def.lower(), 'Visao Q8-B nao identificada'

SQL = {
 'Q3_estados': """
 SELECT nivel, estado, n FROM (
   SELECT 'observacao'::text AS nivel, estado::text AS estado, COUNT(*)::bigint AS n
   FROM derm_v2.observacao WHERE fonte_id=%s GROUP BY estado
   UNION ALL
   SELECT 'catalogo'::text, 'campo_inexistente'::text, COUNT(*)::bigint
   FROM derm_v2.catalogo_fonte_atributo WHERE fonte_id=%s AND campo_existe IS FALSE
 ) t ORDER BY nivel, estado
 """,
 'Q5_prerequisitos': """
 SELECT CASE
  WHEN NOT EXISTS (SELECT 1 FROM derm_v2.paciente WHERE fonte_id=%s AND uf_residencia_ibge IS NOT NULL)
   THEN 'residencia_nao_documentada'
  WHEN NOT EXISTS (SELECT 1 FROM derm_v2.diagnostico WHERE fonte_id=%s)
   THEN 'sem_diagnostico_carregado'
  ELSE 'prerequisitos_disponiveis' END AS status
 """,
 'Q7_divergencias': """
 SELECT COUNT(*)::bigint AS conflitos FROM (
  SELECT o.imagem_id, o.atributo
  FROM derm_v2.observacao o
  JOIN derm_v2.imagem i ON i.imagem_id=o.imagem_id AND i.fonte_id=o.fonte_id
  WHERE o.fonte_id=%s AND o.unidade='imagem' AND o.estado='preenchido'
  GROUP BY o.imagem_id, o.atributo
  HAVING COUNT(DISTINCT COALESCE(o.valor_categoria, o.valor_numerico::text))>1
 ) t
 """,
 'Q8_visao_protegida': """
 SELECT fitzpatrick, pacientes
 FROM analitica_v2.v_cubo_protegido_pad
 WHERE fonte_id=%s ORDER BY fitzpatrick
 """
}
MAP = {'':None, '1.0':'I','2.0':'II','3.0':'III','4.0':'IV','5.0':'V','6.0':'VI'}

# Base original, sem leitura de staging ou das tabelas SQL no oraculo pandas.
def pandas_q(nome):
    if nome=='Q3_estados':
        originais = df['fitspatrick'].fillna('').astype(str).str.strip()
        inexistentes = sum(c not in df.columns for c in ('raca_cor_ibge','ita','monk'))
        return {'catalogo|campo_inexistente':inexistentes,
                'observacao|ausente':int(originais.eq('').sum()),
                'observacao|preenchido':int(originais.ne('').sum())}
    if nome=='Q5_prerequisitos':
        # Apenas compatibilidade do motivo: nao e comparacao de uma consulta clinica.
        if 'uf_residencia_ibge' not in df.columns: return {'status':'residencia_nao_documentada'}
        return {'status':'sem_diagnostico_carregado'}
    if nome=='Q7_divergencias':
        temp = df[['img_id','fitspatrick']].copy()
        temp['classe'] = temp['fitspatrick'].fillna('').astype(str).str.strip().map(MAP)
        assert temp.loc[temp['fitspatrick'].ne(''), 'classe'].notna().all()
        divergencias = temp.groupby('img_id')['classe'].nunique(dropna=True)
        return {'conflitos':int((divergencias>1).sum())}
    if nome=='Q8_visao_protegida':
        por_paciente = defaultdict(lambda: {'valores':set(),'ausencias':0})
        for paciente, original in zip(df['patient_id'], df['fitspatrick']):
            paciente, original = str(paciente).strip(), str(original).strip()
            assert original in MAP, 'Valor de Fitzpatrick nao previsto'
            if original == '': por_paciente[paciente]['ausencias'] += 1
            else: por_paciente[paciente]['valores'].add(MAP[original])
        frequencias = Counter()
        for item in por_paciente.values():
            valores, ausencias = item['valores'], item['ausencias']
            if len(valores)>1: classe='discordante'
            elif len(valores)==1 and ausencias>0: classe='misto_preenchido_ausente'
            elif len(valores)==1: classe=next(iter(valores))
            elif ausencias>0: classe='ausente'
            else: classe='sem_observacao'
            frequencias[classe] += 1
        # Replica a politica Q8-B: suprimir n<5 e a maior categoria se houver primarias.
        maior = sorted(frequencias, key=lambda g:(-frequencias[g],g))[0]
        ha_pequeno = any(v<5 for v in frequencias.values())
        protegido = {g:int(n) for g,n in frequencias.items()
                     if n>=5 and not(ha_pequeno and g==maior)}
        return dict(sorted(protegido.items()))
    raise KeyError(nome)

con = None
papel = 'derm_bench2_'+uuid.uuid4().hex[:12]
senha = secrets.token_urlsafe(24)
papel_criado = False

def quantil(xs, p):
    a=sorted(xs); u=(len(a)-1)*p; i=int(u); j=min(i+1,len(a)-1)
    return a[i]+(a[j]-a[i])*(u-i)

def sumarizar(xs):
    q1,q3 = quantil(xs,.25),quantil(xs,.75)
    return {'n':len(xs),'mediana_ms':statistics.median(xs),'q1_ms':q1,
            'q3_ms':q3,'iiq_ms':q3-q1,'max_ms':max(xs)}

def medir(func,nome):
    inicio=time.perf_counter_ns(); func(nome)
    return (time.perf_counter_ns()-inicio)/1_000_000

def sql_q(nome):
    with con.cursor() as cur:
        params=(fonte,fonte) if nome in ('Q3_estados','Q5_prerequisitos') else (fonte,)
        cur.execute(SQL[nome], params)
        rows=cur.fetchall()
    if nome=='Q3_estados': return {r[0]+'|'+r[1]:int(r[2]) for r in rows}
    if nome=='Q5_prerequisitos': return {'status':rows[0][0]}
    if nome=='Q7_divergencias': return {'conflitos':int(rows[0][0])}
    if nome=='Q8_visao_protegida': return {str(r[0]):int(r[1]) for r in rows}
    raise KeyError(nome)

try:
    admin(f"CREATE ROLE {papel} LOGIN NOSUPERUSER NOCREATEDB NOCREATEROLE PASSWORD '{senha}';")
    papel_criado=True
    admin(f"GRANT USAGE ON SCHEMA derm_v2, analitica_v2 TO {papel}; "
          f"GRANT SELECT ON derm_v2.paciente, derm_v2.diagnostico, derm_v2.imagem, "
          f"derm_v2.observacao, derm_v2.catalogo_fonte_atributo, "
          f"analitica_v2.v_cubo_protegido_pad TO {papel};")
    con=psycopg.connect(host='127.0.0.1',port=5432,dbname='postgres',user=papel,
                       password=senha,autocommit=True,connect_timeout=10)
    with con.cursor() as cur:
        cur.execute('SET default_transaction_read_only=on;')
        cur.execute("SET statement_timeout='60s';")
    saida={
      'experimento':'E5 complementar — Q3/Q5/Q7/Q8', 'sha256_csv':SHA,
      'fonte_id':fonte,
      'ambiente':{'postgresql':versao,'python':platform.python_version(),
                  'pandas':pd.__version__,'psycopg':psycopg.__version__,
                  'cpu_disponiveis':os.cpu_count(), 'csv_linhas':len(df)},
      'protocolo':{'aquecimentos':3,'repeticoes':20,
                   'ordem':'alternada SQL/pandas',
                   'dados':'CSV em memoria e PostgreSQL previamente carregado'},
      'consultas':{},'criterios':{}
    }
    for nome in SQL:
        a,b=sql_q(nome),pandas_q(nome)
        if a!=b: raise AssertionError(f'Divergencia {nome}: SQL={a}; pandas={b}')
        for _ in range(3): medir(sql_q,nome);medir(pandas_q,nome)
        t_sql,t_pd=[],[]
        for i in range(20):
            if i%2==0:
                t_sql.append(medir(sql_q,nome));t_pd.append(medir(pandas_q,nome))
            else:
                t_pd.append(medir(pandas_q,nome));t_sql.append(medir(sql_q,nome))
        r_sql,r_pd=sumarizar(t_sql),sumarizar(t_pd)
        razao=r_sql['mediana_ms']/r_pd['mediana_ms'] if r_pd['mediana_ms']>0 else None
        # EXPLAIN na mesma conexao somente-leitura
        with con.cursor() as cur:
            params=(fonte,fonte) if nome in ('Q3_estados','Q5_prerequisitos') else (fonte,)
            cur.execute('EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) '+SQL[nome],params)
            plano=cur.fetchone()[0]
        saida['consultas'][nome]={
           'resultado_sql_pandas':a,'concordancia_sql_pandas':True,
           'sql':r_sql,'pandas':r_pd,'razao_mediana_sql_sobre_pandas':razao,
           'meta_exploratoria_razao_ate_10':razao is not None and razao<=10,
           'sem_tempo_acima_60s':max(t_sql+t_pd)<=60000,
           'tempos_sql_ms':t_sql,'tempos_pandas_ms':t_pd,
           'explain_analyze_buffers':plano,
           'comparabilidade':('status de indisponibilidade; operacoes diferentes — '
                              'nao interpretar razao como desempenho clinico' if nome=='Q5_prerequisitos'
                              else 'mesma saida e regras por consulta')
        }
        print(f"{nome} | SQL: {r_sql['mediana_ms']:.4f} ms | pandas: {r_pd['mediana_ms']:.4f} ms | razao: {razao:.3f}")
    comparaveis=[n for n in SQL if n!='Q5_prerequisitos']
    saida['criterios']={
      'quatro_saidas_conferidas':len(saida['consultas'])==4,
      'vinte_repeticoes_cada_metodo':all(v['sql']['n']==20 and v['pandas']['n']==20 for v in saida['consultas'].values()),
      'nenhuma_execucao_acima_60s':all(v['sem_tempo_acima_60s'] for v in saida['consultas'].values()),
      'meta_ate_10_nas_tres_consultas_comparaveis':all(saida['consultas'][n]['meta_exploratoria_razao_ate_10'] for n in comparaveis),
      'q5_status_nao_clinico':saida['consultas']['Q5_prerequisitos']['resultado_sql_pandas']=={'status':'residencia_nao_documentada'},
    }
    saida['limites']=('Q5 retorna status de impossibilidade: comparar tempos com pandas nao mede desempenho de consulta clinica. '
                     'Q7 possui zero conflitos naturais; sua deteccao foi avaliada separadamente por injecao controlada. '
                     'Q8 mede visao Q8-B e o oraculo pandas replica supressao primaria e secundaria; '
                     'nao demonstra anonimato completo. Dados ja carregados, mesmo recorte e ambiente local.')
    destino=Path('/content/evidencia_E5_complemento_Q3_Q5_Q7_Q8.json')
    destino.write_text(json.dumps(saida,ensure_ascii=False,indent=2),encoding='utf-8')
    print('CRITERIOS:',json.dumps(saida['criterios'],indent=2,ensure_ascii=False))
    print('Evidencia salva em:',destino)
finally:
    if con is not None: con.close()
    if papel_criado:
        admin(f"REVOKE SELECT ON derm_v2.paciente, derm_v2.diagnostico, derm_v2.imagem, "
              f"derm_v2.observacao, derm_v2.catalogo_fonte_atributo, "
              f"analitica_v2.v_cubo_protegido_pad FROM {papel}; "
              f"REVOKE USAGE ON SCHEMA derm_v2, analitica_v2 FROM {papel}; "
              f"DROP ROLE {papel};")


Q3_estados | SQL: 1.0556 ms | pandas: 2.0062 ms | razao: 0.526
Q5_prerequisitos | SQL: 0.4200 ms | pandas: 0.0083 ms | razao: 50.622
Q7_divergencias | SQL: 3.3332 ms | pandas: 8.2109 ms | razao: 0.406
Q8_visao_protegida | SQL: 7.8044 ms | pandas: 3.1279 ms | razao: 2.495
CRITERIOS: {
  "quatro_saidas_conferidas": true,
  "vinte_repeticoes_cada_metodo": true,
  "nenhuma_execucao_acima_60s": true,
  "meta_ate_10_nas_tres_consultas_comparaveis": true,
  "q5_status_nao_clinico": true
}
Evidencia salva em: /content/evidencia_E5_complemento_Q3_Q5_Q7_Q8.json


In [19]:

import hashlib
import json
from pathlib import Path

# E1 — Inventário de expressividade da PAD-UFES-20

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA
assert len(df.columns) == 26

estruturados = {
    "patient_id": "paciente.identificador_origem",
    "lesion_id": (
        "lesao.identificador_origem, "
        "identidade composta com paciente"
    ),
    "img_id": "imagem.identificador_origem",
    "fitspatrick": (
        "observacao + mapeamento_valor + "
        "catalogo_fonte_atributo"
    )
}

campos = []

for coluna in df.columns:
    valores = df[coluna].fillna("").astype(str).str.strip()

    campos.append({
        "campo": coluna,
        "registros": int(len(valores)),
        "vazios": int(valores.eq("").sum()),
        "UNK": int(valores.str.upper().eq("UNK").sum()),
        "valores_distintos": int(valores.nunique()),
        "preservado_em_staging": True,
        "representacao_estruturada": estruturados.get(coluna),
        "classificacao_E1": "PENDENTE",
        "justificativa_E1": None
    })

relatorio = {
    "experimento": "E1 - inventario inicial de expressividade",
    "fonte": "PAD-UFES-20 metadata.csv",
    "sha256_csv": SHA,
    "total_campos": len(campos),
    "campos_preservados_em_jsonb": len(campos),
    "campos_com_representacao_estruturada_explicita": sum(
        c["representacao_estruturada"] is not None
        for c in campos
    ),
    "campos_apenas_preservados_no_registro_original": sum(
        c["representacao_estruturada"] is None
        for c in campos
    ),
    "avaliacao_semantica_concluida": False,
    "criterio_E1_95_porcento": "NAO_AVALIADO",
    "campos": campos
}

print("INVENTARIO E1")
print("Total de campos:", relatorio["total_campos"])
print(
    "Representacoes estruturadas explicitas:",
    relatorio["campos_com_representacao_estruturada_explicita"]
)
print(
    "Apenas preservados em staging:",
    relatorio["campos_apenas_preservados_no_registro_original"]
)

print("\nCAMPOS A ANALISAR:")
for campo in campos:
    if campo["representacao_estruturada"] is None:
        print(
            "-",
            campo["campo"],
            "| vazios:", campo["vazios"],
            "| UNK:", campo["UNK"]
        )

destino = Path(
    "/content/evidencia_E1_inventario_inicial.json"
)

destino.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nRelatorio salvo:", destino)


INVENTARIO E1
Total de campos: 26
Representacoes estruturadas explicitas: 4
Apenas preservados em staging: 22

CAMPOS A ANALISAR:
- smoke | vazios: 804 | UNK: 0
- drink | vazios: 804 | UNK: 0
- background_father | vazios: 818 | UNK: 95
- background_mother | vazios: 822 | UNK: 84
- age | vazios: 0 | UNK: 0
- pesticide | vazios: 804 | UNK: 0
- gender | vazios: 804 | UNK: 0
- skin_cancer_history | vazios: 804 | UNK: 0
- cancer_history | vazios: 804 | UNK: 0
- has_piped_water | vazios: 804 | UNK: 0
- has_sewage_system | vazios: 804 | UNK: 0
- region | vazios: 0 | UNK: 0
- diameter_1 | vazios: 804 | UNK: 0
- diameter_2 | vazios: 804 | UNK: 0
- diagnostic | vazios: 0 | UNK: 0
- itch | vazios: 0 | UNK: 6
- grew | vazios: 0 | UNK: 402
- hurt | vazios: 0 | UNK: 10
- changed | vazios: 0 | UNK: 396
- bleed | vazios: 0 | UNK: 6
- elevation | vazios: 0 | UNK: 2
- biopsed | vazios: 0 | UNK: 0

Relatorio salvo: /content/evidencia_E1_inventario_inicial.json


In [20]:

import hashlib
import json
from pathlib import Path

# ======================================================
# E1-B — CONSISTENCIA SEMANTICA POR UNIDADE
# PAD-UFES-20 | CSV original
# ======================================================

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA

dados = df.fillna("").copy()

for coluna in dados.columns:
    dados[coluna] = dados[coluna].astype(str).str.strip()

# Classificacao baseada no significado documentado.
# A unidade e CANDIDATA, ainda nao e mapeamento validado.

campos_por_unidade = {
    "paciente": [
        "smoke",
        "drink",
        "background_father",
        "background_mother",
        "age",
        "pesticide",
        "gender",
        "skin_cancer_history",
        "cancer_history",
        "has_piped_water",
        "has_sewage_system"
    ],
    "lesao": [
        "region",
        "diameter_1",
        "diameter_2",
        "diagnostic",
        "itch",
        "grew",
        "hurt",
        "changed",
        "bleed",
        "elevation",
        "biopsed"
    ],
    "fenotipo_por_imagem": [
        "fitspatrick"
    ]
}

chaves = {
    "paciente": ["patient_id"],
    "lesao": ["patient_id", "lesion_id"],
    "fenotipo_por_imagem": ["img_id"]
}

identificadores = {
    "patient_id", "lesion_id", "img_id"
}

campos_classificados = (
    identificadores
    | {
        campo
        for grupo in campos_por_unidade.values()
        for campo in grupo
    }
)

assert campos_classificados == set(dados.columns), (
    "Ha campos sem classificacao ou campos inesperados."
)

assert len(dados) == 2298

def contagem_por_grupo(mascara, keys, indice):
    auxiliar = dados[keys].copy()
    auxiliar["_flag"] = mascara.astype(int)

    contagens = (
        auxiliar.groupby(keys, sort=False)["_flag"]
        .sum()
    )

    return contagens.reindex(
        indice, fill_value=0
    )

def analisar_campo(campo, unidade):
    keys = chaves[unidade]

    grupos = dados.groupby(keys, sort=False).size()

    valores = dados[campo]

    vazio = valores.eq("")
    desconhecido = valores.str.upper().eq("UNK")
    conhecido = ~(vazio | desconhecido)

    distintos_conhecidos = (
        dados.loc[conhecido, keys + [campo]]
        .groupby(keys, sort=False)[campo]
        .nunique()
        .reindex(grupos.index, fill_value=0)
    )

    n_vazios = contagem_por_grupo(
        vazio, keys, grupos.index
    )

    n_desconhecidos = contagem_por_grupo(
        desconhecido, keys, grupos.index
    )

    n_conhecidos = contagem_por_grupo(
        conhecido, keys, grupos.index
    )

    return {
        "campo": campo,
        "unidade_candidata": unidade,
        "total_unidades": int(len(grupos)),
        "unidades_com_multiplas_linhas": int(
            (grupos > 1).sum()
        ),
        "linhas_vazias": int(vazio.sum()),
        "linhas_UNK": int(desconhecido.sum()),
        "unidades_com_valores_conhecidos_distintos": int(
            (distintos_conhecidos > 1).sum()
        ),
        "unidades_com_vazio_e_valor_conhecido": int(
            ((n_vazios > 0) & (n_conhecidos > 0)).sum()
        ),
        "unidades_com_UNK_e_valor_conhecido": int(
            (
                (n_desconhecidos > 0)
                & (n_conhecidos > 0)
            ).sum()
        ),
        "status_mapeamento": (
            "ja_estruturado"
            if campo == "fitspatrick"
            else "pendente"
        )
    }

resultados = []

for unidade, campos in campos_por_unidade.items():
    for campo in campos:
        resultados.append(
            analisar_campo(campo, unidade)
        )

# Verificacao adicional:
# o fototipo esta armazenado por imagem;
# avalia-se tambem sua consistencia entre
# imagens do mesmo paciente, SEM alterar
# a granularidade do modelo.

chaves["fenotipo_no_paciente"] = ["patient_id"]

verificacao_fenotipo = analisar_campo(
    "fitspatrick",
    "fenotipo_no_paciente"
)

relatorio = {
    "experimento": (
        "E1-B - consistencia por unidade observacional"
    ),
    "sha256_csv": SHA,
    "linhas_csv": len(dados),
    "total_campos_csv": len(dados.columns),
    "campos_avaliados": len(resultados),
    "identificadores": sorted(identificadores),
    "analises": resultados,
    "fitzpatrick_entre_imagens_do_mesmo_paciente": (
        verificacao_fenotipo
    ),
    "mapeamento_dos_22_campos_concluido": False,
    "criterio_E1_95_porcento": "NAO_AVALIADO",
    "observacao": (
        "Divergencia entre registros nao implica "
        "automaticamente erro: pode refletir "
        "coletas distintas, ausencias ou variacao "
        "do atributo. Nenhum valor foi alterado."
    )
}

print("E1-B — CONSISTENCIA POR UNIDADE")
print("Linhas:", len(dados))
print("Campos analisados:", len(resultados))

print("\nRESULTADOS POR CAMPO")
for item in resultados:
    print(
        f"{item['campo']:22s} "
        f"| unidade={item['unidade_candidata']:18s} "
        f"| valores_distintos_na_unidade="
        f"{item['unidades_com_valores_conhecidos_distintos']:4d} "
        f"| vazio+preenchido="
        f"{item['unidades_com_vazio_e_valor_conhecido']:4d} "
        f"| UNK+preenchido="
        f"{item['unidades_com_UNK_e_valor_conhecido']:4d}"
    )

print("\nFITZPATRICK ENTRE IMAGENS DO MESMO PACIENTE")
print(json.dumps(
    verificacao_fenotipo,
    ensure_ascii=False,
    indent=2
))

destino = Path(
    "/content/evidencia_E1_consistencia_semantica.json"
)

destino.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nRelatorio salvo:", destino)


E1-B — CONSISTENCIA POR UNIDADE
Linhas: 2298
Campos analisados: 23

RESULTADOS POR CAMPO
smoke                  | unidade=paciente           | valores_distintos_na_unidade=   0 | vazio+preenchido=   0 | UNK+preenchido=   0
drink                  | unidade=paciente           | valores_distintos_na_unidade=   0 | vazio+preenchido=   0 | UNK+preenchido=   0
background_father      | unidade=paciente           | valores_distintos_na_unidade=   0 | vazio+preenchido=   0 | UNK+preenchido=   0
background_mother      | unidade=paciente           | valores_distintos_na_unidade=   0 | vazio+preenchido=   0 | UNK+preenchido=   0
age                    | unidade=paciente           | valores_distintos_na_unidade=  90 | vazio+preenchido=   0 | UNK+preenchido=   0
pesticide              | unidade=paciente           | valores_distintos_na_unidade=   0 | vazio+preenchido=   0 | UNK+preenchido=   0
gender                 | unidade=paciente           | valores_distintos_na_unidade=   0 | vazio+preenchido=

In [21]:

import hashlib
import json
import subprocess
import pandas as pd
from pathlib import Path

# ===================================================
# E1-C — DOMINIOS DOS METADADOS E IDADE
# PAD-UFES-20 | pandas versus PostgreSQL 16
# ===================================================

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA

dados = df.fillna("").copy()

for coluna in dados.columns:
    dados[coluna] = (
        dados[coluna].astype(str).str.strip()
    )

assert len(dados) == 2298
assert len(dados.columns) == 26

# ---------------------------------------------------
# 1. CONFERIR IDADES POR PACIENTE
# ---------------------------------------------------

idades_por_paciente = (
    dados.groupby("patient_id")["age"]
    .nunique()
)

pacientes_com_idades_distintas = int(
    (idades_por_paciente > 1).sum()
)

assert pacientes_com_idades_distintas == 90

distribuicao_idades = {
    str(k): int(v)
    for k, v in (
        idades_por_paciente
        .value_counts()
        .sort_index()
        .items()
    )
}

# Converter apenas para análise.
# O valor original permanece inalterado.

idades_numericas = pd.to_numeric(
    dados["age"].str.replace(",", ".", regex=False),
    errors="coerce"
)

nao_numericas = int(
    idades_numericas.isna().sum()
)

auxiliar = pd.DataFrame({
    "patient_id": dados["patient_id"],
    "idade_numerica": idades_numericas
})

amplitude = (
    auxiliar.groupby("patient_id")["idade_numerica"]
    .agg(lambda serie: (
        serie.max() - serie.min()
        if serie.notna().any()
        else float("nan")
    ))
)

amplitude_divergentes = amplitude[
    idades_por_paciente > 1
]

# ---------------------------------------------------
# 2. CONFERENCIA INDEPENDENTE NO POSTGRESQL
# ---------------------------------------------------

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

sql = f"""
SELECT COUNT(*)
FROM (
    SELECT
        s.registro_jsonb ->> 'patient_id'
            AS paciente,

        COUNT(DISTINCT
            BTRIM(s.registro_jsonb ->> 'age')
        ) AS idades_distintas

    FROM derm_v2.staging_registro s

    JOIN derm_v2.fonte f
      ON f.fonte_id = s.fonte_id

    WHERE f.nome = 'PAD-UFES-20 metadata.csv'
      AND f.versao = 'sha256:{SHA}'

    GROUP BY
        s.registro_jsonb ->> 'patient_id'

    HAVING COUNT(DISTINCT
        BTRIM(s.registro_jsonb ->> 'age')
    ) > 1
) AS pacientes_divergentes;
"""

r = subprocess.run(
    PSQL,
    input=sql,
    text=True,
    capture_output=True
)

assert r.returncode == 0, r.stderr

pacientes_sql = int(r.stdout.strip())

# ---------------------------------------------------
# 3. INVENTARIO DOS 22 CAMPOS
# ---------------------------------------------------

campos_estruturados = {
    "patient_id",
    "lesion_id",
    "img_id",
    "fitspatrick"
}

inventario = []

for coluna in dados.columns:
    if coluna in campos_estruturados:
        continue

    valores = dados[coluna]

    vazios = valores.eq("")
    desconhecidos = valores.str.upper().eq("UNK")

    conhecidos = valores[
        ~(vazios | desconhecidos)
    ]

    inventario.append({
        "campo": coluna,
        "vazios": int(vazios.sum()),
        "UNK": int(desconhecidos.sum()),
        "valores_conhecidos_distintos": int(
            conhecidos.nunique()
        ),
        "status": "mapeamento_semantico_pendente"
    })

# ---------------------------------------------------
# 4. CRITERIOS E RELATORIO
# ---------------------------------------------------

criterios = {
    "csv_original_confirmado": True,
    "pandas_90_pacientes": (
        pacientes_com_idades_distintas == 90
    ),
    "sql_pandas_concordam": (
        pacientes_sql == pacientes_com_idades_distintas
    ),
    "todos_os_22_campos_inventariados": (
        len(inventario) == 22
    )
}

def numero_ou_nulo(x):
    return (
        float(x)
        if pd.notna(x)
        else None
    )

relatorio = {
    "experimento": (
        "E1-C - caracterizacao de idade e dominios"
    ),
    "sha256_csv": SHA,
    "total_pacientes": int(
        dados["patient_id"].nunique()
    ),
    "pacientes_com_idades_distintas_pandas": (
        pacientes_com_idades_distintas
    ),
    "pacientes_com_idades_distintas_sql": (
        pacientes_sql
    ),
    "distribuicao_quantidade_idades": (
        distribuicao_idades
    ),
    "idades_nao_numericas": nao_numericas,
    "amplitude_idade_nos_pacientes_divergentes": {
        "minima": numero_ou_nulo(
            amplitude_divergentes.min()
        ),
        "mediana": numero_ou_nulo(
            amplitude_divergentes.median()
        ),
        "maxima": numero_ou_nulo(
            amplitude_divergentes.max()
        )
    },
    "inventario_campos": inventario,
    "criterios": criterios,
    "todos_atendidos": all(criterios.values()),
    "nota": (
        "Diferencas de idade nao sao classificadas "
        "automaticamente como erros. "
        "Nenhum identificador individual foi "
        "incluido no relatorio."
    )
}

print("RESULTADO E1-C")
print(json.dumps(
    {
        chave: valor
        for chave, valor in relatorio.items()
        if chave != "inventario_campos"
    },
    ensure_ascii=False,
    indent=2
))

print("\nDOMINIOS DOS 22 CAMPOS")
for campo in inventario:
    print(
        campo["campo"],
        "| conhecidos distintos:",
        campo["valores_conhecidos_distintos"],
        "| vazios:", campo["vazios"],
        "| UNK:", campo["UNK"]
    )

destino = Path(
    "/content/evidencia_E1C_dominios_idade.json"
)

destino.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nRelatorio salvo:", destino)

assert relatorio["todos_atendidos"]

print("\nE1-C CONCLUIDO.")


RESULTADO E1-C
{
  "experimento": "E1-C - caracterizacao de idade e dominios",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "total_pacientes": 1373,
  "pacientes_com_idades_distintas_pandas": 90,
  "pacientes_com_idades_distintas_sql": 90,
  "distribuicao_quantidade_idades": {
    "1": 1283,
    "2": 88,
    "3": 2
  },
  "idades_nao_numericas": 0,
  "amplitude_idade_nos_pacientes_divergentes": {
    "minima": 1.0,
    "mediana": 1.0,
    "maxima": 30.0
  },
  "criterios": {
    "csv_original_confirmado": true,
    "pandas_90_pacientes": true,
    "sql_pandas_concordam": true,
    "todos_os_22_campos_inventariados": true
  },
  "todos_atendidos": true,
  "nota": "Diferencas de idade nao sao classificadas automaticamente como erros. Nenhum identificador individual foi incluido no relatorio."
}

DOMINIOS DOS 22 CAMPOS
smoke | conhecidos distintos: 2 | vazios: 804 | UNK: 0
drink | conhecidos distintos: 2 | vazios: 804 | UNK: 0
background_father | c

In [23]:

# ============================================================
# E1-D CORRIGIDO — MAPEAMENTO DOS 22 CAMPOS DA PAD-UFES-20
# PostgreSQL 16 | Google Colab
# Seguro para reexecucao: nao duplica registros existentes
# ============================================================

import hashlib
import json
import subprocess
from pathlib import Path

SHA = (
    "14d145235cedb022548257acb0d84dcd949e2"
    "c916f65d2baa7c38ed5339e9527"
)

assert hashlib.sha256(conteudo).hexdigest() == SHA
assert len(df) == 2298
assert len(df.columns) == 26

PACIENTE = [
    "smoke",
    "drink",
    "background_father",
    "background_mother",
    "age",
    "pesticide",
    "gender",
    "skin_cancer_history",
    "cancer_history",
    "has_piped_water",
    "has_sewage_system"
]

LESAO = [
    "region",
    "diameter_1",
    "diameter_2",
    "diagnostic",
    "itch",
    "grew",
    "hurt",
    "changed",
    "bleed",
    "elevation",
    "biopsed"
]

NUMERICOS = {
    "age",
    "diameter_1",
    "diameter_2"
}

assert len(PACIENTE) == 11
assert len(LESAO) == 11

assert set(PACIENTE + LESAO) == (
    set(df.columns) - {
        "patient_id",
        "lesion_id",
        "img_id",
        "fitspatrick"
    }
)

# ------------------------------------------------------------
# 1. CONEXAO POSTGRESQL
# ------------------------------------------------------------

PSQL = [
    "runuser", "-u", "postgres", "--",
    "psql", "-X", "-q", "-t", "-A",
    "-v", "ON_ERROR_STOP=1",
    "-d", "postgres"
]

def executar(comando):
    p = subprocess.run(
        PSQL,
        input=comando,
        text=True,
        capture_output=True
    )

    if p.returncode != 0:
        raise RuntimeError(p.stderr)

    return p.stdout.strip()

versao = executar("SHOW server_version;")

assert versao.startswith("16."), versao

fonte = executar(f"""
SELECT fonte_id
FROM derm_v2.fonte
WHERE nome = 'PAD-UFES-20 metadata.csv'
  AND versao = 'sha256:{SHA}';
""").splitlines()

assert len(fonte) == 1

FONTE = int(fonte[0])

print("PostgreSQL:", versao)
print("Fonte PAD-UFES-20:", FONTE)

# ------------------------------------------------------------
# 2. VERIFICAR SE A EXTENSAO JA EXISTE
# ------------------------------------------------------------

estado_tabelas = executar("""
SELECT
    (
        to_regclass(
            'derm_v2.catalogo_campo_adicional'
        ) IS NOT NULL
    )::int,

    (
        to_regclass(
            'derm_v2.valor_campo_adicional'
        ) IS NOT NULL
    )::int;
""")

assert estado_tabelas in ("0|0", "1|1"), (
    "Extensao parcialmente instalada. "
    f"Estado encontrado: {estado_tabelas}"
)

# ------------------------------------------------------------
# 3. PREPARAR CATALOGO
# ------------------------------------------------------------

linhas_catalogo = []

for campo in PACIENTE:
    linhas_catalogo.append((
        campo,
        "paciente",
        (
            "numerico_candidato"
            if campo in NUMERICOS
            else "categoria_original"
        )
    ))

for campo in LESAO:
    linhas_catalogo.append((
        campo,
        "lesao",
        (
            "numerico_candidato"
            if campo in NUMERICOS
            else "categoria_original"
        )
    ))

valores_sql = ",\n".join(
    "('%s','%s','%s')" % item
    for item in linhas_catalogo
)

# ------------------------------------------------------------
# 4. INSTALAR EXTENSAO, SOMENTE SE NECESSARIO
# ------------------------------------------------------------

migracao = f"""
BEGIN;

CREATE TABLE derm_v2.catalogo_campo_adicional (
    fonte_id bigint NOT NULL
        REFERENCES derm_v2.fonte(fonte_id),

    campo text NOT NULL,

    unidade_candidata text NOT NULL
        CHECK (
            unidade_candidata IN (
                'paciente', 'lesao'
            )
        ),

    natureza_candidata text NOT NULL
        CHECK (
            natureza_candidata IN (
                'categoria_original',
                'numerico_candidato'
            )
        ),

    harmonizacao_validada boolean
        NOT NULL DEFAULT false,

    PRIMARY KEY (fonte_id, campo),

    CONSTRAINT uq_campo_unidade
        UNIQUE (
            fonte_id,
            campo,
            unidade_candidata
        )
);

CREATE TABLE derm_v2.valor_campo_adicional (
    fonte_id bigint NOT NULL,
    staging_id bigint NOT NULL,
    imagem_id bigint NOT NULL,
    paciente_id bigint NOT NULL,
    lesao_id bigint,

    campo text NOT NULL,

    unidade_candidata text NOT NULL
        CHECK (
            unidade_candidata IN (
                'paciente', 'lesao'
            )
        ),

    valor_original text NOT NULL,

    estado text NOT NULL
        CHECK (
            estado IN (
                'preenchido',
                'ausente',
                'desconhecido'
            )
        ),

    valor_numerico_candidato numeric,

    PRIMARY KEY (
        fonte_id,
        staging_id,
        campo
    ),

    FOREIGN KEY (
        fonte_id,
        campo,
        unidade_candidata
    )
    REFERENCES derm_v2.catalogo_campo_adicional (
        fonte_id,
        campo,
        unidade_candidata
    ),

    FOREIGN KEY (
        staging_id,
        fonte_id
    )
    REFERENCES derm_v2.staging_registro (
        staging_id,
        fonte_id
    ),

    FOREIGN KEY (
        imagem_id,
        fonte_id
    )
    REFERENCES derm_v2.imagem (
        imagem_id,
        fonte_id
    ),

    FOREIGN KEY (
        paciente_id,
        fonte_id
    )
    REFERENCES derm_v2.paciente (
        paciente_id,
        fonte_id
    ),

    FOREIGN KEY (
        lesao_id,
        fonte_id
    )
    REFERENCES derm_v2.lesao (
        lesao_id,
        fonte_id
    ),

    CHECK (
        (
            unidade_candidata = 'paciente'
            AND lesao_id IS NULL
        )
        OR
        (
            unidade_candidata = 'lesao'
            AND lesao_id IS NOT NULL
        )
    ),

    CHECK (
        (
            estado = 'ausente'
            AND btrim(valor_original) = ''
        )
        OR
        (
            estado = 'desconhecido'
            AND upper(btrim(valor_original)) = 'UNK'
        )
        OR
        (
            estado = 'preenchido'
            AND btrim(valor_original) <> ''
            AND upper(btrim(valor_original)) <> 'UNK'
        )
    )
);

CREATE INDEX ix_valor_adicional_campo_estado
ON derm_v2.valor_campo_adicional (
    fonte_id, campo, estado
);

CREATE INDEX ix_valor_adicional_paciente
ON derm_v2.valor_campo_adicional (
    fonte_id, paciente_id, campo
);

CREATE INDEX ix_valor_adicional_lesao
ON derm_v2.valor_campo_adicional (
    fonte_id, lesao_id, campo
)
WHERE lesao_id IS NOT NULL;

INSERT INTO derm_v2.catalogo_campo_adicional (
    fonte_id,
    campo,
    unidade_candidata,
    natureza_candidata
)
SELECT
    {FONTE},
    x.campo,
    x.unidade,
    x.natureza
FROM (
    VALUES {valores_sql}
) AS x(campo, unidade, natureza);

DO $$
BEGIN
    IF EXISTS (
        SELECT 1
        FROM derm_v2.staging_registro s
        CROSS JOIN derm_v2.catalogo_campo_adicional c
        WHERE s.fonte_id = {FONTE}
          AND c.fonte_id = {FONTE}
          AND NOT (
              s.registro_jsonb ? c.campo
          )
    ) THEN
        RAISE EXCEPTION
            'Campos ausentes no JSONB de staging';
    END IF;
END $$;

INSERT INTO derm_v2.valor_campo_adicional (
    fonte_id,
    staging_id,
    imagem_id,
    paciente_id,
    lesao_id,
    campo,
    unidade_candidata,
    valor_original,
    estado,
    valor_numerico_candidato
)
SELECT
    s.fonte_id,
    s.staging_id,
    i.imagem_id,
    p.paciente_id,

    CASE
        WHEN c.unidade_candidata = 'lesao'
        THEN l.lesao_id
        ELSE NULL
    END,

    c.campo,
    c.unidade_candidata,

    s.registro_jsonb ->> c.campo,

    CASE
        WHEN btrim(
            s.registro_jsonb ->> c.campo
        ) = ''
        THEN 'ausente'

        WHEN upper(btrim(
            s.registro_jsonb ->> c.campo
        )) = 'UNK'
        THEN 'desconhecido'

        ELSE 'preenchido'
    END,

    CASE
        WHEN
            c.natureza_candidata =
                'numerico_candidato'
            AND
            btrim(
                s.registro_jsonb ->> c.campo
            ) ~ '^[0-9]+([.,][0-9]+)?$'

        THEN replace(
            btrim(
                s.registro_jsonb ->> c.campo
            ),
            ',', '.'
        )::numeric

        ELSE NULL
    END

FROM derm_v2.staging_registro s

JOIN derm_v2.imagem i
    ON i.fonte_id = s.fonte_id
   AND i.staging_id = s.staging_id

JOIN derm_v2.lesao l
    ON l.fonte_id = i.fonte_id
   AND l.lesao_id = i.lesao_id

JOIN derm_v2.paciente p
    ON p.fonte_id = l.fonte_id
   AND p.paciente_id = l.paciente_id

JOIN derm_v2.catalogo_campo_adicional c
    ON c.fonte_id = s.fonte_id

WHERE s.fonte_id = {FONTE};

DO $$
DECLARE
    total bigint;
    erros bigint;
BEGIN
    SELECT COUNT(*)
    INTO total
    FROM derm_v2.valor_campo_adicional
    WHERE fonte_id = {FONTE};

    IF total <> 50556 THEN
        RAISE EXCEPTION
            'Total incorreto de observacoes: %',
            total;
    END IF;

    SELECT COUNT(*)
    INTO erros
    FROM derm_v2.valor_campo_adicional v

    JOIN derm_v2.staging_registro s
      ON s.staging_id = v.staging_id
     AND s.fonte_id = v.fonte_id

    JOIN derm_v2.imagem i
      ON i.imagem_id = v.imagem_id
     AND i.fonte_id = v.fonte_id

    JOIN derm_v2.lesao l
      ON l.lesao_id = i.lesao_id
     AND l.fonte_id = i.fonte_id

    WHERE v.fonte_id = {FONTE}
      AND (
          v.valor_original IS DISTINCT FROM
              s.registro_jsonb ->> v.campo

          OR i.staging_id IS DISTINCT FROM
              v.staging_id

          OR l.paciente_id IS DISTINCT FROM
              v.paciente_id

          OR (
              v.unidade_candidata = 'lesao'
              AND v.lesao_id IS DISTINCT FROM
                  l.lesao_id
          )
      );

    IF erros <> 0 THEN
        RAISE EXCEPTION
            'Erros de rastreabilidade: %',
            erros;
    END IF;
END $$;

COMMIT;
"""

if estado_tabelas == "0|0":
    print("Instalando a extensao E1-D...")

    executar(migracao)

    print("Extensao instalada com sucesso.")

else:
    print(
        "Extensao E1-D ja instalada. "
        "Reutilizando os registros existentes."
    )

# ------------------------------------------------------------
# 5. CONSULTA DE VALIDACAO — CORRIGIDA
# ------------------------------------------------------------

consulta = f"""
SELECT json_build_object(

    'total_observacoes',
    (
        SELECT COUNT(*)
        FROM derm_v2.valor_campo_adicional
        WHERE fonte_id = {FONTE}
    ),

    'campos_catalogados',
    (
        SELECT COUNT(*)
        FROM derm_v2.catalogo_campo_adicional
        WHERE fonte_id = {FONTE}
    ),

    'idades_divergentes',
    (
        SELECT COUNT(*)
        FROM (
            SELECT paciente_id

            FROM derm_v2.valor_campo_adicional

            WHERE fonte_id = {FONTE}
              AND campo = 'age'
              AND estado = 'preenchido'

            GROUP BY paciente_id

            HAVING COUNT(
                DISTINCT btrim(valor_original)
            ) > 1

        ) AS pacientes_divergentes
    ),

    'resumo',
    (
        SELECT json_agg(row_to_json(x))

        FROM (
            SELECT
                campo,
                COUNT(*) AS registros,

                COUNT(*) FILTER (
                    WHERE estado = 'ausente'
                ) AS vazios,

                COUNT(*) FILTER (
                    WHERE estado = 'desconhecido'
                ) AS unk,

                COUNT(*) FILTER (
                    WHERE estado = 'preenchido'
                      AND valor_numerico_candidato
                          IS NULL
                      AND campo IN (
                          'age',
                          'diameter_1',
                          'diameter_2'
                      )
                ) AS numericos_sem_parse

            FROM derm_v2.valor_campo_adicional

            WHERE fonte_id = {FONTE}

            GROUP BY campo
            ORDER BY campo

        ) AS x
    )
);
"""

resultado = json.loads(
    executar(consulta)
)

# ------------------------------------------------------------
# 6. CONFERENCIA INDEPENDENTE COM PANDAS
# ------------------------------------------------------------

referencia = df.fillna("").astype(str)

comparacoes = []

for item in resultado["resumo"]:

    campo = item["campo"]

    valores = referencia[campo].str.strip()

    esperado = {
        "registros": int(len(valores)),
        "vazios": int(
            valores.eq("").sum()
        ),
        "unk": int(
            valores.str.upper().eq("UNK").sum()
        )
    }

    obtido = {
        chave: int(item[chave])
        for chave in esperado
    }

    comparacoes.append({
        "campo": campo,
        "confere": obtido == esperado,
        "sql": obtido,
        "pandas": esperado,
        "numericos_sem_parse": int(
            item["numericos_sem_parse"]
        )
    })

# ------------------------------------------------------------
# 7. CRITERIOS DE ACEITACAO
# ------------------------------------------------------------

criterios = {
    "22_campos_catalogados":
        int(resultado["campos_catalogados"]) == 22,

    "50556_observacoes_por_registro":
        int(resultado["total_observacoes"]) == 50556,

    "contagens_22_campos_coincidem":
        (
            len(comparacoes) == 22
            and all(
                x["confere"]
                for x in comparacoes
            )
        ),

    "90_pacientes_com_idades_distintas_preservados":
        int(resultado["idades_divergentes"]) == 90
}

# ------------------------------------------------------------
# 8. GERAR EVIDENCIA
# ------------------------------------------------------------

relatorio = {
    "experimento":
        "E1-D - representacao de campos adicionais",

    "postgresql": versao,

    "sha256_csv": SHA,

    "natureza": (
        "Extensao estrutural com proveniencia. "
        "Harmonizacao semantica ainda nao concluida."
    ),

    "resultados": {
        chave: valor
        for chave, valor in resultado.items()
        if chave != "resumo"
    },

    "por_campo": comparacoes,

    "criterios": criterios,

    "todos_atendidos": all(
        criterios.values()
    ),

    "E1_95_porcento": (
        "NAO_AVALIADO: preservacao estruturada "
        "nao equivale a harmonizacao semantica."
    )
}

saida = Path(
    "/content/evidencia_E1D_campos_adicionais.json"
)

saida.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

resumo_final = {
    "experimento": relatorio["experimento"],

    "resultados": relatorio["resultados"],

    "criterios": criterios,

    "campos_numericos_nao_convertidos": {
        x["campo"]: x["numericos_sem_parse"]
        for x in comparacoes
        if x["campo"] in NUMERICOS
    },

    "todos_atendidos": relatorio["todos_atendidos"]
}

print("\nRESULTADO E1-D")

print(json.dumps(
    resumo_final,
    ensure_ascii=False,
    indent=2
))

print("\nRelatorio salvo:", saida)

assert relatorio["todos_atendidos"], (
    "Algum criterio falhou. "
    "Verifique o relatorio JSON."
)

print("\nE1-D CONCLUIDO COM SUCESSO.")


PostgreSQL: 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)
Fonte PAD-UFES-20: 1
Extensao E1-D ja instalada. Reutilizando os registros existentes.

RESULTADO E1-D
{
  "experimento": "E1-D - representacao de campos adicionais",
  "resultados": {
    "total_observacoes": 50556,
    "campos_catalogados": 22,
    "idades_divergentes": 90
  },
  "criterios": {
    "22_campos_catalogados": true,
    "50556_observacoes_por_registro": true,
    "contagens_22_campos_coincidem": true,
    "90_pacientes_com_idades_distintas_preservados": true
  },
  "campos_numericos_nao_convertidos": {
    "age": 0,
    "diameter_1": 0,
    "diameter_2": 0
  },
  "todos_atendidos": true
}

Relatorio salvo: /content/evidencia_E1D_campos_adicionais.json

E1-D CONCLUIDO COM SUCESSO.


In [24]:
# E1-E — auditoria de rastreabilidade, estados e dominios; somente leitura
import hashlib, json, subprocess
from pathlib import Path
import pandas as pd

SHA = '14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527'
assert hashlib.sha256(conteudo).hexdigest() == SHA
assert len(df) == 2298 and len(df.columns) == 26

CAMPOS = [
    'smoke','drink','background_father','background_mother','age',
    'pesticide','gender','skin_cancer_history','cancer_history',
    'has_piped_water','has_sewage_system','region','diameter_1',
    'diameter_2','diagnostic','itch','grew','hurt','changed',
    'bleed','elevation','biopsed'
]
NUMERICOS = {'age','diameter_1','diameter_2'}
assert len(CAMPOS) == 22

PSQL = ['runuser','-u','postgres','--','psql','-X','-q','-t','-A',
        '-v','ON_ERROR_STOP=1','-d','postgres']
def executar(query):
    p = subprocess.run(PSQL, input=query, text=True, capture_output=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr)
    return p.stdout.strip()

ids = executar(f"""
SELECT fonte_id FROM derm_v2.fonte
WHERE nome='PAD-UFES-20 metadata.csv' AND versao='sha256:{SHA}';
""").splitlines()
assert len(ids) == 1
fonte_id = int(ids[0])

sql = f"""
WITH auditoria AS (
 SELECT v.campo, v.valor_original, v.estado, v.valor_numerico_candidato,
        c.natureza_candidata,
        s.registro_jsonb ->> v.campo AS valor_staging,
        (s.registro_jsonb ? v.campo) AS campo_existe,
        i.staging_id AS staging_da_imagem,
        i.lesao_id AS lesao_da_imagem,
        l.paciente_id AS paciente_da_lesao,
        v.staging_id, v.lesao_id, v.paciente_id,
        CASE WHEN btrim(v.valor_original)='' THEN 'ausente'
             WHEN upper(btrim(v.valor_original))='UNK' THEN 'desconhecido'
             ELSE 'preenchido' END AS estado_esperado,
        CASE WHEN c.natureza_candidata='numerico_candidato'
                  AND btrim(v.valor_original) ~ '^[0-9]+([.,][0-9]+)?$'
             THEN replace(btrim(v.valor_original),',','.')::numeric
             ELSE NULL END AS numerico_esperado
 FROM derm_v2.valor_campo_adicional v
 JOIN derm_v2.catalogo_campo_adicional c
   ON c.fonte_id=v.fonte_id AND c.campo=v.campo
 JOIN derm_v2.staging_registro s
   ON s.staging_id=v.staging_id AND s.fonte_id=v.fonte_id
 JOIN derm_v2.imagem i
   ON i.imagem_id=v.imagem_id AND i.fonte_id=v.fonte_id
 JOIN derm_v2.lesao l
   ON l.lesao_id=i.lesao_id AND l.fonte_id=i.fonte_id
 WHERE v.fonte_id={fonte_id}
)
SELECT json_build_object(
 'total_valores',count(*),
 'divergencias_valor',count(*) FILTER (WHERE valor_original IS DISTINCT FROM valor_staging),
 'chaves_ausentes_staging',count(*) FILTER (WHERE NOT campo_existe),
 'divergencias_estado',count(*) FILTER (WHERE estado IS DISTINCT FROM estado_esperado),
 'divergencias_numericas',count(*) FILTER (WHERE valor_numerico_candidato IS DISTINCT FROM numerico_esperado),
 'divergencias_vinculo',count(*) FILTER (
     WHERE staging_id IS DISTINCT FROM staging_da_imagem
        OR paciente_id IS DISTINCT FROM paciente_da_lesao
        OR (lesao_id IS NOT NULL AND lesao_id IS DISTINCT FROM lesao_da_imagem)
        OR (lesao_id IS NULL AND campo IN (
          'region','diameter_1','diameter_2','diagnostic','itch','grew','hurt',
          'changed','bleed','elevation','biopsed'))
 ),
 'campos_validacao_semantica_pendente',
    (SELECT COUNT(*) FROM derm_v2.catalogo_campo_adicional
     WHERE fonte_id={fonte_id} AND NOT harmonizacao_validada),
 'campos_catalogados',
    (SELECT COUNT(*) FROM derm_v2.catalogo_campo_adicional
     WHERE fonte_id={fonte_id}),
 'linhas_staging',
    (SELECT COUNT(*) FROM derm_v2.staging_registro WHERE fonte_id={fonte_id}),
 'pacientes_idades_distintas',
    (SELECT COUNT(*) FROM (
       SELECT paciente_id FROM derm_v2.valor_campo_adicional
       WHERE fonte_id={fonte_id} AND campo='age' AND estado='preenchido'
       GROUP BY paciente_id HAVING COUNT(DISTINCT valor_original)>1
    ) z)
) FROM auditoria;
"""
resultado = json.loads(executar(sql))

# Comparacao independente: estados, dominios e cobertura a partir do CSV original
ref = df.fillna('').astype(str)
por_campo = []
for campo in CAMPOS:
    v = ref[campo].str.strip()
    ausente = v.eq('')
    desconhecido = v.str.upper().eq('UNK')
    conhecido = ~(ausente | desconhecido)
    entry = {
        'campo': campo,
        'total_linhas': int(len(v)),
        'ausentes': int(ausente.sum()),
        'desconhecidos_UNK': int(desconhecido.sum()),
        'preenchidos': int(conhecido.sum()),
        'dominio_valores_conhecidos_distintos': int(v[conhecido].nunique()),
        'tipo_candidato': 'numerico' if campo in NUMERICOS else 'categorico_original'
    }
    if campo in NUMERICOS:
        nums = pd.to_numeric(v[conhecido].str.replace(',', '.', regex=False), errors='coerce')
        entry['nao_convertiveis'] = int(nums.isna().sum())
        entry['minimo_observado'] = float(nums.min()) if nums.notna().any() else None
        entry['maximo_observado'] = float(nums.max()) if nums.notna().any() else None
        entry['unidade_fisica_validada'] = False
    else:
        entry['categorias_originais'] = sorted(v[conhecido].unique().tolist())
        entry['mapeamento_de_significado_validado'] = False
    por_campo.append(entry)

criterios = {
    '50556_valores': resultado['total_valores'] == 2298 * 22,
    '22_campos_catalogados': resultado['campos_catalogados'] == 22,
    '2298_linhas_staging': resultado['linhas_staging'] == 2298,
    'valores_conferem_com_staging': resultado['divergencias_valor'] == 0,
    'campos_presentes_no_staging': resultado['chaves_ausentes_staging'] == 0,
    'estados_conferem': resultado['divergencias_estado'] == 0,
    'conversao_numerica_coerente': resultado['divergencias_numericas'] == 0,
    'vinculos_conferem': resultado['divergencias_vinculo'] == 0,
    'idades_distintas_preservadas': resultado['pacientes_idades_distintas'] == 90,
    'todos_numericos_convertiveis': all(
        x.get('nao_convertiveis', 0) == 0 for x in por_campo
    ),
    'sem_harmonizacao_semantica_falsamente_declarada':
        resultado['campos_validacao_semantica_pendente'] == 22,
}
relatorio = {
    'experimento': 'E1-E - fidelidade e auditoria preliminar de dominios',
    'sha256_csv': SHA,
    'resultados_sql': resultado,
    'dominios_origem_pandas': por_campo,
    'criterios': criterios,
    'todos_atendidos': all(criterios.values()),
    'meta_95_semantica': 'NAO_AVALIADA',
    'observacao': 'Tipos e unidades sao candidatos. Listas originais nao sao mapeamento clinico validado.'
}
saida = Path('/content/evidencia_E1E_auditoria_dominios.json')
saida.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding='utf-8')
print('E1-E - RESULTADOS SQL')
print(json.dumps(resultado, ensure_ascii=False, indent=2))
print('\nCRITERIOS')
print(json.dumps(criterios, ensure_ascii=False, indent=2))
print('\nDOMINIOS CATEGORICOS (SEM IDs INDIVIDUAIS)')
for d in por_campo:
    if 'categorias_originais' in d:
        print(f"{d['campo']}: {d['categorias_originais']}")
    else:
        print(f"{d['campo']}: numerico, minimo={d['minimo_observado']}, maximo={d['maximo_observado']}, nao_convertiveis={d['nao_convertiveis']}")
print('\nRelatorio salvo:',saida)
print('E1-E CONCLUIDO' if relatorio['todos_atendidos'] else 'E1-E: CRITERIOS NAO ATENDIDOS')


E1-E - RESULTADOS SQL
{
  "total_valores": 50556,
  "divergencias_valor": 0,
  "chaves_ausentes_staging": 0,
  "divergencias_estado": 0,
  "divergencias_numericas": 0,
  "divergencias_vinculo": 0,
  "campos_validacao_semantica_pendente": 22,
  "campos_catalogados": 22,
  "linhas_staging": 2298,
  "pacientes_idades_distintas": 90
}

CRITERIOS
{
  "50556_valores": true,
  "22_campos_catalogados": true,
  "2298_linhas_staging": true,
  "valores_conferem_com_staging": true,
  "campos_presentes_no_staging": true,
  "estados_conferem": true,
  "conversao_numerica_coerente": true,
  "vinculos_conferem": true,
  "idades_distintas_preservadas": true,
  "todos_numericos_convertiveis": true,
  "sem_harmonizacao_semantica_falsamente_declarada": true
}

DOMINIOS CATEGORICOS (SEM IDs INDIVIDUAIS)
smoke: ['False', 'True']
drink: ['False', 'True']
background_father: ['AUSTRIA', 'BRASIL', 'BRAZIL', 'CZECH', 'GERMANY', 'ISRAEL', 'ITALY', 'NETHERLANDS', 'POLAND', 'POMERANIA', 'PORTUGAL', 'SPAIN']
backgro

In [25]:
# E1-F — Tipagem semântica documentada e auditoria de domínios da fonte
# PAD-UFES-20 / PostgreSQL 16 / Google Colab
# Preserva dados originais; reexecutável; não declara harmonização interfontes.
import hashlib
import json
import subprocess
from pathlib import Path
import pandas as pd

SHA = '14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527'
DOC = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7479321/'
assert hashlib.sha256(conteudo).hexdigest() == SHA
assert len(df) == 2298 and len(df.columns) == 26

# Campo, unidade candidata, tipo lógico, significado, unidade de medida.
# Unidades físicas dos diâmetros não foram especificadas no dicionário citado.
REGRAS = [
    ('smoke', 'paciente', 'booleano', 'Uso de cigarro registrado', None),
    ('drink', 'paciente', 'booleano', 'Consumo de bebida alcoólica registrado', None),
    ('background_father', 'paciente', 'categoria_origem', 'Ascendência paterna declarada, país ou região', None),
    ('background_mother', 'paciente', 'categoria_origem', 'Ascendência materna declarada, país ou região', None),
    ('age', 'paciente', 'inteiro', 'Idade registrada na observação de origem', 'anos'),
    ('pesticide', 'paciente', 'booleano', 'Uso de pesticidas registrado', None),
    ('gender', 'paciente', 'categoria_controlada', 'Gênero registrado na fonte', None),
    ('skin_cancer_history', 'paciente', 'booleano', 'Histórico de câncer de pele do paciente ou familiar', None),
    ('cancer_history', 'paciente', 'booleano', 'Histórico de câncer do paciente ou familiar', None),
    ('has_piped_water', 'paciente', 'booleano', 'Acesso domiciliar a água encanada', None),
    ('has_sewage_system', 'paciente', 'booleano', 'Acesso domiciliar a esgotamento sanitário', None),
    ('region', 'lesao', 'categoria_controlada', 'Macrorregião anatômica da lesão, não residência', None),
    ('diameter_1', 'lesao', 'decimal', 'Diâmetro horizontal da lesão', None),
    ('diameter_2', 'lesao', 'decimal', 'Diâmetro vertical da lesão', None),
    ('diagnostic', 'lesao', 'categoria_controlada', 'Código de diagnóstico na fonte', None),
    ('itch', 'lesao', 'booleano', 'Presença de prurido', None),
    ('grew', 'lesao', 'booleano', 'Crescimento recente da lesão', None),
    ('hurt', 'lesao', 'booleano', 'Presença de dor', None),
    ('changed', 'lesao', 'booleano', 'Mudança recente da lesão', None),
    ('bleed', 'lesao', 'booleano', 'Sangramento da lesão', None),
    ('elevation', 'lesao', 'booleano', 'Elevação da lesão', None),
    ('biopsed', 'lesao', 'booleano', 'Confirmação diagnóstica por biópsia (True) ou consenso clínico (False)', None),
]
DOMINIOS = {
    'gender': {'FEMALE', 'MALE'},
    'region': {'ABDOMEN','ARM','BACK','CHEST','EAR','FACE','FOOT','FOREARM',
               'HAND','LIP','NECK','NOSE','SCALP','SHIN','THIGH'},
    'diagnostic': {'ACK','BCC','MEL','NEV','SCC','SEK'},
}
assert len(REGRAS) == 22
assert len({r[0] for r in REGRAS}) == 22
assert set(df.columns) - {r[0] for r in REGRAS} == {
    'patient_id', 'lesion_id', 'img_id', 'fitspatrick'
}

PSQL = ['runuser','-u','postgres','--','psql','-X','-q','-t','-A',
        '-v','ON_ERROR_STOP=1','-d','postgres']
def executar(q):
    p = subprocess.run(PSQL, input=q, text=True, capture_output=True)
    if p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout.strip()

def lit(v):
    return 'NULL' if v is None else "'" + str(v).replace("'", "''") + "'"

fonte = executar(f"""
SELECT fonte_id FROM derm_v2.fonte
WHERE nome='PAD-UFES-20 metadata.csv' AND versao='sha256:{SHA}';
""").splitlines()
assert len(fonte) == 1
FONTE = int(fonte[0])

valores_sql = ',\n'.join(
    '(' + ','.join([str(FONTE)] + [lit(x) for x in regra] + [lit(DOC)]) + ')'
    for regra in REGRAS
)

# Um catálogo explícito e uma view tipada. Os 50.556 registros continuam intactos.
# ON CONFLICT não sobrescreve regras antigas: eventual diferença gera falha.
migracao = f"""
BEGIN;
CREATE TABLE IF NOT EXISTS derm_v2.regra_semantica_adicional (
    fonte_id bigint NOT NULL,
    campo text NOT NULL,
    unidade_candidata text NOT NULL,
    tipo_logico text NOT NULL CHECK (tipo_logico IN
        ('booleano','categoria_origem','categoria_controlada','inteiro','decimal')),
    significado text NOT NULL,
    unidade_medida text,
    referencia_documental text NOT NULL,
    PRIMARY KEY (fonte_id, campo),
    FOREIGN KEY (fonte_id, campo, unidade_candidata)
        REFERENCES derm_v2.catalogo_campo_adicional
        (fonte_id, campo, unidade_candidata)
);
INSERT INTO derm_v2.regra_semantica_adicional
    (fonte_id,campo,unidade_candidata,tipo_logico,significado,unidade_medida,referencia_documental)
VALUES {valores_sql}
ON CONFLICT (fonte_id,campo) DO NOTHING;

DO $$
BEGIN
    IF (SELECT COUNT(*) FROM derm_v2.regra_semantica_adicional
        WHERE fonte_id={FONTE}) <> 22 THEN
        RAISE EXCEPTION 'Quantidade incorreta de regras semanticas';
    END IF;
    IF EXISTS (
        WITH esperado (fonte_id,campo,unidade_candidata,tipo_logico,significado,unidade_medida,referencia_documental)
          AS (VALUES {valores_sql})
        SELECT * FROM esperado
        EXCEPT
        SELECT fonte_id,campo,unidade_candidata,tipo_logico,significado,unidade_medida,referencia_documental
        FROM derm_v2.regra_semantica_adicional WHERE fonte_id={FONTE}
    ) THEN
        RAISE EXCEPTION 'Regras ja existentes diferentes da especificacao E1-F';
    END IF;
END $$;

CREATE OR REPLACE VIEW derm_v2.v_valor_tipado_e1f AS
SELECT
    v.fonte_id, v.staging_id, v.imagem_id,
    v.paciente_id, v.lesao_id,
    v.campo, v.valor_original, v.estado,
    r.tipo_logico, r.significado, r.unidade_medida,
    CASE WHEN v.estado='preenchido' AND r.tipo_logico='booleano'
        THEN CASE v.valor_original WHEN 'True' THEN true
             WHEN 'False' THEN false ELSE NULL END
    END AS valor_booleano,
    CASE WHEN v.estado='preenchido' AND r.tipo_logico='inteiro'
        THEN v.valor_numerico_candidato::bigint END AS valor_inteiro,
    CASE WHEN v.estado='preenchido' AND r.tipo_logico='decimal'
        THEN v.valor_numerico_candidato END AS valor_decimal,
    CASE WHEN v.estado='preenchido' AND r.tipo_logico IN
        ('categoria_controlada','categoria_origem')
        THEN v.valor_original END AS valor_categoria
FROM derm_v2.valor_campo_adicional v
JOIN derm_v2.regra_semantica_adicional r
  ON r.fonte_id=v.fonte_id AND r.campo=v.campo;
COMMIT;
"""
executar(migracao)

# Conferência SQL dos códigos documentados e dos campos tipados.
regioes = ','.join(lit(x) for x in sorted(DOMINIOS['region']))
diagnosticos = ','.join(lit(x) for x in sorted(DOMINIOS['diagnostic']))
q = f"""
WITH ver AS (
 SELECT *,
    CASE
    WHEN estado <> 'preenchido' THEN false
    WHEN tipo_logico='booleano' THEN valor_original NOT IN ('True','False')
    WHEN tipo_logico='inteiro' THEN
        valor_inteiro IS NULL OR valor_inteiro < 0 OR
        valor_inteiro::numeric IS DISTINCT FROM
        (SELECT valor_numerico_candidato FROM derm_v2.valor_campo_adicional z
         WHERE z.fonte_id=v.fonte_id AND z.staging_id=v.staging_id AND z.campo=v.campo)
    WHEN tipo_logico='decimal' THEN valor_decimal IS NULL OR valor_decimal < 0
    WHEN campo='gender' THEN valor_categoria NOT IN ('FEMALE','MALE')
    WHEN campo='region' THEN valor_categoria NOT IN ({regioes})
    WHEN campo='diagnostic' THEN valor_categoria NOT IN ({diagnosticos})
    WHEN tipo_logico='categoria_origem' THEN valor_categoria IS NULL
    ELSE true END AS dominio_invalido,
    CASE
    WHEN estado <> 'preenchido' THEN false
    WHEN tipo_logico='booleano' THEN valor_booleano IS NULL
    WHEN tipo_logico='inteiro' THEN valor_inteiro IS NULL
    WHEN tipo_logico='decimal' THEN valor_decimal IS NULL
    WHEN tipo_logico IN ('categoria_controlada','categoria_origem') THEN valor_categoria IS NULL
    ELSE true END AS sem_valor_tipado
 FROM derm_v2.v_valor_tipado_e1f v
 WHERE fonte_id={FONTE}
)
SELECT json_build_object(
 'registros_tipados', COUNT(*),
 'campos_tipados', COUNT(DISTINCT campo),
 'preenchidos', COUNT(*) FILTER (WHERE estado='preenchido'),
 'ausentes', COUNT(*) FILTER (WHERE estado='ausente'),
 'desconhecidos', COUNT(*) FILTER (WHERE estado='desconhecido'),
 'dominios_invalidos', COUNT(*) FILTER (WHERE dominio_invalido),
 'preenchidos_sem_valor_tipado', COUNT(*) FILTER (WHERE sem_valor_tipado),
 'regras_cadastradas', (SELECT COUNT(*) FROM derm_v2.regra_semantica_adicional
                       WHERE fonte_id={FONTE}),
 'harmonizacao_interfontes_declarada',
    (SELECT COUNT(*) FROM derm_v2.catalogo_campo_adicional
      WHERE fonte_id={FONTE} AND harmonizacao_validada)
) FROM ver;
"""
resultado = json.loads(executar(q))

# Auditoria independente dos códigos da fonte no pandas.
ref = df.fillna('').astype(str)
falhas_pandas = {}
total_preenchidos = 0
total_ausentes = 0
total_desconhecidos = 0
for campo, unidade, tipo, significado, medida in REGRAS:
    valores = ref[campo].str.strip()
    ausentes = valores.eq('')
    desconhecidos = valores.str.upper().eq('UNK')
    preenchidos = ~(ausentes | desconhecidos)
    conhecidos = valores[preenchidos]
    total_ausentes += int(ausentes.sum())
    total_desconhecidos += int(desconhecidos.sum())
    total_preenchidos += int(preenchidos.sum())

    if tipo == 'booleano':
        invalidos = int((~conhecidos.isin({'True','False'})).sum())
    elif tipo == 'inteiro':
        numericos = pd.to_numeric(conhecidos, errors='coerce')
        invalidos = int((numericos.isna() | (numericos < 0) |
                         (numericos % 1 != 0)).sum())
    elif tipo == 'decimal':
        numericos = pd.to_numeric(
            conhecidos.str.replace(',', '.', regex=False), errors='coerce'
        )
        invalidos = int((numericos.isna() | (numericos < 0)).sum())
    elif campo in DOMINIOS:
        invalidos = int((~conhecidos.isin(DOMINIOS[campo])).sum())
    else:
        invalidos = 0  # Origem ancestral preservada, sem converter para país.
    falhas_pandas[campo] = invalidos

criterios = {
    '50556_valores_tipados': resultado['registros_tipados'] == 50556,
    '22_campos_com_regras': resultado['campos_tipados'] == 22 and resultado['regras_cadastradas'] == 22,
    'sem_dominios_invalidos_sql': resultado['dominios_invalidos'] == 0,
    'sem_dominios_invalidos_pandas': all(n == 0 for n in falhas_pandas.values()),
    'sem_preenchidos_sem_tipo': resultado['preenchidos_sem_valor_tipado'] == 0,
    'estados_conferem_com_pandas': (
        resultado['preenchidos'] == total_preenchidos
        and resultado['ausentes'] == total_ausentes
        and resultado['desconhecidos'] == total_desconhecidos
    ),
    'sem_afirmar_harmonizacao_interfontes': resultado['harmonizacao_interfontes_declarada'] == 0,
}
relatorio = {
    'experimento': 'E1-F - tipagem documentada de 22 atributos',
    'referencia_dicionario': DOC,
    'sha256_csv': SHA,
    'fonte_id': FONTE,
    'tipos_logicos': {r[0]: r[2] for r in REGRAS},
    'unidades_nao_documentadas': ['diameter_1', 'diameter_2'],
    'resultado_sql': resultado,
    'dominios_invalidos_pandas_por_campo': falhas_pandas,
    'criterios': criterios,
    'todos_atendidos': all(criterios.values()),
    'meta_95_semantica_interfontes': 'NAO_AVALIADA',
    'nota': ('Confere significado e tipos na fonte, sem equivalência com ontologias externas. '
             'POMERANIA permanece região ancestral; BRASIL e BRAZIL não são mesclados; '
             'diâmetros não recebem unidade física presumida; idades não são consolidadas.')
}
caminho = Path('/content/evidencia_E1F_tipagem_semantica.json')
caminho.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding='utf-8')
print('RESULTADO E1-F')
print(json.dumps({k:v for k,v in relatorio.items() if k not in ('tipos_logicos','dominios_invalidos_pandas_por_campo')},
                 ensure_ascii=False, indent=2))
print('\nRelatorio completo:', caminho)
assert relatorio['todos_atendidos'], 'Critérios não atendidos: consultar relatório'
print('\nE1-F CONCLUIDO.')


RESULTADO E1-F
{
  "experimento": "E1-F - tipagem documentada de 22 atributos",
  "referencia_dicionario": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7479321/",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "fonte_id": 1,
  "unidades_nao_documentadas": [
    "diameter_1",
    "diameter_2"
  ],
  "resultado_sql": {
    "registros_tipados": 50556,
    "campos_tipados": 22,
    "preenchidos": 39875,
    "ausentes": 9680,
    "desconhecidos": 1001,
    "dominios_invalidos": 0,
    "preenchidos_sem_valor_tipado": 0,
    "regras_cadastradas": 22,
    "harmonizacao_interfontes_declarada": 0
  },
  "criterios": {
    "50556_valores_tipados": true,
    "22_campos_com_regras": true,
    "sem_dominios_invalidos_sql": true,
    "sem_dominios_invalidos_pandas": true,
    "sem_preenchidos_sem_tipo": true,
    "estados_conferem_com_pandas": true,
    "sem_afirmar_harmonizacao_interfontes": true
  },
  "todos_atendidos": true,
  "meta_95_semantica_interfontes": "

In [26]:
RESULTADO E1-F
{
  "experimento": "E1-F - tipagem documentada de 22 atributos",
  "referencia_dicionario": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7479321/",
  "sha256_csv": "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527",
  "fonte_id": 1,
  "unidades_nao_documentadas": [
    "diameter_1",
    "diameter_2"
  ],
  "resultado_sql": {
    "registros_tipados": 50556,
    "campos_tipados": 22,
    "preenchidos": 39875,
    "ausentes": 9680,
    "desconhecidos": 1001,
    "dominios_invalidos": 0,
    "preenchidos_sem_valor_tipado": 0,
    "regras_cadastradas": 22,
    "harmonizacao_interfontes_declarada": 0
  },
  "criterios": {
    "50556_valores_tipados": true,
    "22_campos_com_regras": true,
    "sem_dominios_invalidos_sql": true,
    "sem_dominios_invalidos_pandas": true,
    "sem_preenchidos_sem_tipo": true,
    "estados_conferem_com_pandas": true,
    "sem_afirmar_harmonizacao_interfontes": true
  },
  "todos_atendidos": true,
  "meta_95_semantica_interfontes": "NAO_AVALIADA",
  "nota": "Confere significado e tipos na fonte, sem equivalência com ontologias externas. POMERANIA permanece região ancestral; BRASIL e BRAZIL não são mesclados; diâmetros não recebem unidade física presumida; idades não são consolidadas."
}

Relatorio completo: /content/evidencia_E1F_tipagem_semantica.json

E1-F CONCLUIDO.


SyntaxError: invalid syntax (3074124978.py, line 1)

In [27]:
# E2 final — 8 testes negativos A-H do esquema derm_v2 no PostgreSQL 16.
# Replica o teste SQL de restricoes v2.1 já previsto, agora com evidência estruturada.
# Executa transação sintetica e ROLLBACK; nunca escreve na PAD-UFES-20.
# Executar no MESMO Google Colab do experimento.

import hashlib
import json
import re
import subprocess
from pathlib import Path

SHA = "14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527"
assert hashlib.sha256(conteudo).hexdigest() == SHA

PSQL = ["runuser", "-u", "postgres", "--", "psql", "-X", "-q", "-t", "-A", "-v", "ON_ERROR_STOP=1", "-d", "postgres"]

def executar(consulta, permitir_erro=False):
    p = subprocess.run(PSQL, input=consulta, text=True, capture_output=True)
    if p.returncode and not permitir_erro:
        raise RuntimeError(p.stderr + "\n" + p.stdout)
    return p

versao = executar("SHOW server_version;").stdout.strip()
assert versao.startswith("16."), versao

fonte_query = f"SELECT fonte_id FROM derm_v2.fonte WHERE nome='PAD-UFES-20 metadata.csv' AND versao='sha256:{SHA}';"
fonte_raw = executar(fonte_query).stdout.strip().splitlines()
assert len(fonte_raw) == 1
fonte = int(fonte_raw[0])

# O teste usa fixtures em fontes novas que serao revertidas.
# A leitura anterior/posterior verifica as tabelas da PAD e a extensao E1.
def contar_pad():
    q = f"""
    SELECT json_build_object(
      'staging', (SELECT COUNT(*) FROM derm_v2.staging_registro WHERE fonte_id={fonte}),
      'pacientes', (SELECT COUNT(*) FROM derm_v2.paciente WHERE fonte_id={fonte}),
      'lesoes', (SELECT COUNT(*) FROM derm_v2.lesao WHERE fonte_id={fonte}),
      'imagens', (SELECT COUNT(*) FROM derm_v2.imagem WHERE fonte_id={fonte}),
      'observacoes', (SELECT COUNT(*) FROM derm_v2.observacao WHERE fonte_id={fonte}),
      'campos_adicionais', (SELECT COUNT(*) FROM derm_v2.catalogo_campo_adicional WHERE fonte_id={fonte}),
      'valores_adicionais', (SELECT COUNT(*) FROM derm_v2.valor_campo_adicional WHERE fonte_id={fonte}),
      'regras_semanticas', (SELECT COUNT(*) FROM derm_v2.regra_semantica_adicional WHERE fonte_id={fonte}),
      'fontes_fixture_persistidas', (SELECT COUNT(*) FROM derm_v2.fonte WHERE nome IN ('FONTE_SINTETICA_A', 'FONTE_SINTETICA_B'))
    );
    """
    return json.loads(executar(q).stdout.strip())

antes = contar_pad()
assert antes == {
    'staging': 2298,
    'pacientes': 1373,
    'lesoes': 1891,
    'imagens': 2298,
    'observacoes': 2298,
    'campos_adicionais': 22,
    'valores_adicionais': 50556,
    'regras_semanticas': 22,
    'fontes_fixture_persistidas': 0
}, antes

print("PostgreSQL:", versao)
print("Executando testes negativos A-H em uma transacao com ROLLBACK...")

script_sql = r"""
-- TESTE DE FUMACA SINTETICO PARA POSTGRESQL 16.
-- Pre-requisito: esquema_postgresql16_v2.sql instalado sem erros.
-- NAO executar em producao. ROLLBACK ao final, sem persistencia.
-- Resultados nao foram produzidos pelo autor deste arquivo.
BEGIN;
SET LOCAL search_path TO derm_v2, pg_catalog;

DO $$
DECLARE
    f1 bigint; f2 bigint;
    p1 bigint; p2 bigint; l1 bigint; i1 bigint; i2 bigint;
    m1 bigint; o1 bigint;
    erro_esperado boolean;
BEGIN
    INSERT INTO fonte(nome, versao, instituicao_fornecedora,
                      uf_instituicao_ibge, municipio_instituicao_ibge)
    VALUES ('FONTE_SINTETICA_A', 'v1', 'INSTITUICAO_FICTICIA', '23', '2304400')
    RETURNING fonte_id INTO f1;
    INSERT INTO fonte(nome, versao) VALUES ('FONTE_SINTETICA_B', 'v1')
    RETURNING fonte_id INTO f2;

    INSERT INTO catalogo_fonte_atributo(fonte_id, atributo, unidade,
                                       campo_existe, nome_campo_origem, tipo_dado_original)
    VALUES (f1, 'raca_cor_ibge', 'paciente', true, 'rotulo_teste_raca', 'text'),
           (f1, 'fitzpatrick', 'imagem', true, 'rotulo_teste_fitz', 'text'),
           (f2, 'monk', 'imagem', false, NULL, NULL);

    INSERT INTO paciente(fonte_id, identificador_origem,
                         uf_residencia_ibge, municipio_residencia_ibge)
    VALUES (f1, 'PACIENTE_TESTE_1', '24', NULL)
    RETURNING paciente_id INTO p1;
    INSERT INTO paciente(fonte_id, identificador_origem)
    VALUES (f2, 'PACIENTE_TESTE_2') RETURNING paciente_id INTO p2;

    INSERT INTO lesao(fonte_id, paciente_id,
                      evidencia_vinculo_paciente, identificador_origem)
    VALUES (f1, p1, 'Ligacao documentada no fixture sintetico', 'LESAO_TESTE_1')
    RETURNING lesao_id INTO l1;
    INSERT INTO imagem(fonte_id, lesao_id, evidencia_vinculo_lesao,
                       modalidade, identificador_origem)
    VALUES (f1, l1, 'Ligacao documentada no fixture sintetico',
            'clinica', 'IMAGEM_TESTE_1') RETURNING imagem_id INTO i1;
    INSERT INTO imagem(fonte_id, modalidade, identificador_origem)
    VALUES (f2, 'dermatoscopica', 'IMAGEM_TESTE_2')
    RETURNING imagem_id INTO i2;

    INSERT INTO mapeamento_valor (
        fonte_id, atributo, unidade, versao_mapeamento,
        valor_origem, estado_destino, valor_categoria_destino, regra_documentada
    ) VALUES (
        f1, 'raca_cor_ibge', 'paciente', 1,
        'Branca', 'preenchido', 'branca', 'Normalizar capitalizacao; fixture sintetico'
    ) RETURNING mapeamento_id INTO m1;

    INSERT INTO observacao (
        fonte_id, atributo, unidade, paciente_id, estado, valor_categoria,
        valor_origem, mapeamento_id, metodo, situacao_atribuidor
    ) VALUES (
        f1, 'raca_cor_ibge', 'paciente', p1, 'preenchido', 'branca',
        'Branca', m1, 'autodeclaracao', 'nao_documentado'
    ) RETURNING observacao_id INTO o1;

    -- Teste A: mapeamento diverge do valor original -> erro de CHECK (23514).
    erro_esperado := false;
    BEGIN
      INSERT INTO observacao(fonte_id, atributo, unidade, paciente_id,
                            estado, valor_categoria, valor_origem, mapeamento_id)
      VALUES (f1, 'raca_cor_ibge', 'paciente', p1,
              'preenchido', 'branca', 'Parda', m1);
    EXCEPTION WHEN check_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'A: mapeamento divergente nao rejeitado'; END IF;
    RAISE NOTICE 'E2_A_PASS';

    -- Teste B: dominio de raca/cor invalido -> erro CHECK.
    erro_esperado := false;
    BEGIN
      INSERT INTO mapeamento_valor(fonte_id, atributo, unidade, versao_mapeamento,
              valor_origem, estado_destino, valor_categoria_destino, regra_documentada)
      VALUES (f1, 'raca_cor_ibge', 'paciente', 1, 'AZUL', 'preenchido', 'azul', 'fixture');
    EXCEPTION WHEN check_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'B: dominio invalido nao rejeitado'; END IF;
    RAISE NOTICE 'E2_B_PASS';

    -- Teste C: associacao entre fontes diferentes -> FK rejeitada.
    erro_esperado := false;
    BEGIN
      INSERT INTO lesao(fonte_id, paciente_id, evidencia_vinculo_paciente)
      VALUES (f2, p1, 'tentativa sintetica');
    EXCEPTION WHEN foreign_key_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'C: FK cruzada nao rejeitada'; END IF;
    RAISE NOTICE 'E2_C_PASS';

    -- Teste D: campo inexistente no catalogo nao aceita observacao.
    erro_esperado := false;
    BEGIN
      INSERT INTO observacao(fonte_id, atributo, unidade, imagem_id, estado)
      VALUES (f2, 'monk', 'imagem', i2, 'ausente');
    EXCEPTION WHEN check_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'D: campo estruturalmente inexistente aceito'; END IF;
    RAISE NOTICE 'E2_D_PASS';

    -- Teste E: paciente E imagem na mesma observacao -> exclusividade rejeitada.
    erro_esperado := false;
    BEGIN
      INSERT INTO observacao(fonte_id, atributo, unidade, paciente_id,
              imagem_id, estado)
      VALUES (f1, 'raca_cor_ibge', 'paciente', p1, i1, 'ausente');
    EXCEPTION WHEN check_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'E: XOR nao aplicado'; END IF;
    RAISE NOTICE 'E2_E_PASS';

    -- Teste F: raca/cor nao deve ser atribuida por algoritmo.
    erro_esperado := false;
    BEGIN
      INSERT INTO observacao(fonte_id, atributo, unidade, paciente_id, estado,
          valor_categoria, valor_origem, mapeamento_id, metodo)
      VALUES (f1, 'raca_cor_ibge', 'paciente', p1, 'preenchido',
              'branca', 'Branca', m1, 'atribuicao_algoritmica');
    EXCEPTION WHEN check_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'F: raca algoritmica aceita'; END IF;
    RAISE NOTICE 'E2_F_PASS';

    -- Teste G: mapeamento nao deve ser editado.
    erro_esperado := false;
    BEGIN
      UPDATE mapeamento_valor SET regra_documentada = 'nova regra' WHERE mapeamento_id = m1;
    EXCEPTION WHEN object_not_in_prerequisite_state THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'G: mapeamento editavel'; END IF;
    RAISE NOTICE 'E2_G_PASS';

    -- Teste H: alternancia do catalogo com dado referenciado deve ser rejeitada.
    erro_esperado := false;
    BEGIN
      UPDATE catalogo_fonte_atributo
      SET campo_existe = false, nome_campo_origem = NULL, tipo_dado_original = NULL
      WHERE fonte_id = f1 AND atributo = 'raca_cor_ibge' AND unidade = 'paciente';
    EXCEPTION WHEN check_violation THEN erro_esperado := true;
    END;
    IF NOT erro_esperado THEN RAISE EXCEPTION 'H: catalogo foi desativado com dados'; END IF;
    RAISE NOTICE 'E2_H_PASS';

    RAISE NOTICE 'Fixtures sinteticos: testes A-H passaram nesta transacao (sera revertida).';
END;
$$;
ROLLBACK;

"""

execucao = executar(script_sql, permitir_erro=True)
marcadores = sorted(set(re.findall(r"E2_([A-H])_PASS", execucao.stderr)))
# A ausência do ROLLBACK pode ser confundida com falha sem efeitos, então
# exigimos que psql tenha chegado ao fim do roteiro e retornado zero.
terminou = execucao.returncode == 0

# Capturamos depois inclusive quando há erro: o banco revoga a transacao
# da sessao desconectada, pois não existe COMMIT no roteiro.
depois = contar_pad()

criterios = {
    f"{letra}_{nome}": (letra in marcadores)
    for letra, nome in [
        ("A", "mapeamento_divergente_rejeitado"),
        ("B", "dominio_invalido_rejeitado"),
        ("C", "associacao_entre_fontes_rejeitada"),
        ("D", "atributo_inexistente_rejeitado"),
        ("E", "sujeito_xor_rejeitado"),
        ("F", "raca_algoritmica_rejeitada"),
        ("G", "mapeamento_imutavel"),
        ("H", "catalogo_protegido_com_dados")
    ]
}
criterios["roteiro_sql_executado_ate_rollback"] = terminou
criterios["pad_e_extensao_inalteradas"] = antes == depois
criterios["nenhuma_fixture_persistida"] = depois["fontes_fixture_persistidas"] == 0

relatorio = {
    "experimento": "E2 — restricoes negativas A-H, transacao sintetica com ROLLBACK",
    "postgresql": versao,
    "sha256_csv": SHA,
    "marcadores": marcadores,
    "returncode_sql": execucao.returncode,
    "pad_antes": antes,
    "pad_depois": depois,
    "criterios": criterios,
    "todos_atendidos": all(criterios.values()),
    "limites": "As oito restricoes testadas não cobrem todos os mecanismos do esquema; resultados limitados aos cenarios sinteticos A-H. ROLLBACK não restaura valores de sequencias identity, mas remove as linhas temporarias.",
    "erro_sql": execucao.stderr[-3000:] if execucao.returncode else None
}

caminho = Path("/content/evidencia_E2_restricoes_A_H.json")
caminho.write_text(json.dumps(relatorio, indent=2, ensure_ascii=False), encoding="utf-8")

print("RESULTADO E2 — TESTES NEGATIVOS A-H")
print(json.dumps({k: relatorio[k] for k in ["marcadores", "criterios", "todos_atendidos"]}, indent=2, ensure_ascii=False))
print("Relatorio salvo:", caminho)
if not relatorio["todos_atendidos"]:
    print("Detalhes do SQL:", execucao.stderr[-3000:])
    raise AssertionError("Um ou mais testes E2 falharam; veja o JSON e o erro SQL.")
print("E2 CONCLUIDO COM SUCESSO.")


PostgreSQL: 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)
Executando testes negativos A-H em uma transacao com ROLLBACK...
RESULTADO E2 — TESTES NEGATIVOS A-H
{
  "marcadores": [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G",
    "H"
  ],
  "criterios": {
    "A_mapeamento_divergente_rejeitado": true,
    "B_dominio_invalido_rejeitado": true,
    "C_associacao_entre_fontes_rejeitada": true,
    "D_atributo_inexistente_rejeitado": true,
    "E_sujeito_xor_rejeitado": true,
    "F_raca_algoritmica_rejeitada": true,
    "G_mapeamento_imutavel": true,
    "H_catalogo_protegido_com_dados": true,
    "roteiro_sql_executado_ate_rollback": true,
    "pad_e_extensao_inalteradas": true,
    "nenhuma_fixture_persistida": true
  },
  "todos_atendidos": true
}
Relatorio salvo: /content/evidencia_E2_restricoes_A_H.json
E2 CONCLUIDO COM SUCESSO.
